# 과제 3 · 사람이 많이 사는 곳은 지하철도 가까울까?

**연구 질문:** 서울시 행정동별 인구밀도와 지하철 접근성은 어떤 관계이며, 고밀도·저접근성 지역은 어디인가?

- 분석 단위: 100m 인구 격자 → 서울 426개 행정동.
- 주 지표: 전체 행정동 면적 기준 인구밀도, 인구가중 최근접역 중력형 접근성(보행망). 절대 기준: 최근접역 1km 밖 인구와 인구 절반 이상이 1km 밖인 동. 보조: 주거·상업·공업지역 면적 밀도, 주민가중 격자 밀도, 14시 생활인구 밀도.
- 범위: OSM 보행망 최단거리 기반 탐색 분석(직선거리는 비교용). 경사·신호·출입구를 반영한 실제 보행시간과 전체 대중교통 접근성은 측정하지 않습니다.
- 자료: 2024 공개 인구, 2025 행정동·격자, 기준일 확인 못한 수업 제공 역 좌표, 파일명상 2026-09 기준 용도지역(도시지역), 2025 생활인구(수업 7주차 파일), 수집일을 기록한 OSM 보행망. 2026년 현황으로 해석하지 않습니다.
- 원본 수업 노트북을 수정하지 않습니다. OSM 보행망만 Overpass에서 한 번 내려받아 `assignment3/.cache/`에 저장하고, 이후에는 캐시를 읽습니다. 그 밖의 API 호출은 없습니다.
- 코드 저장소: https://github.com/ijin5/smartcity-assignment3-transit-access
- Python 커널: `강의자료/.venv/bin/python`. 위에서 아래로 모두 실행합니다.
- 산출물은 이 노트북과 같은 `assignment3` 폴더의 `outputs/`, `dist/`, `REPORT.md`에 저장합니다.

[과제 지시문](../../lecture05_Urban_Planning_Index/03_accessibility.ipynb) · [강의계획서](https://app.notion.com/p/2155235c90d080bda32fc6d654786ce2)

## 0. 라이브러리와 경로

In [ ]:
from pathlib import Path
from datetime import datetime
from zoneinfo import ZoneInfo
import hashlib
import json
import unicodedata

import geopandas as gpd
import numpy as np
import pandas as pd
from scipy.spatial import cKDTree
from scipy.stats import spearmanr, pearsonr

ROOT = Path.cwd().resolve()
if ROOT.name != "assignment3":
    ROOT = ROOT / "강의자료/assignment/assignment3"
if not ROOT.is_dir():
    raise FileNotFoundError("노트북 폴더 또는 과목 작업 폴더에서 실행하세요.")
SOURCE = ROOT.parents[1] / "lecture05_Urban_Planning_Index"
OUT = ROOT / "outputs"
DIST = ROOT / "dist"
CRS = "EPSG:5179"
GU = dict(zip([f"11{i:03d}" for i in range(10, 251, 10)],
    "종로구 중구 용산구 성동구 광진구 동대문구 중랑구 성북구 강북구 도봉구 노원구 은평구 서대문구 마포구 양천구 강서구 구로구 금천구 영등포구 동작구 관악구 서초구 강남구 송파구 강동구".split()))


from IPython.display import display, Image, Markdown
import os


## 1. 자료 읽기와 기준 좌표계

거리·면적 계산에 미터 단위 EPSG:5179를 사용합니다. 경계·격자의 유효성과 키 중복을 먼저 확인합니다.

In [ ]:
OUT.mkdir(exist_ok=True)
DIST.mkdir(exist_ok=True)
population_path = next((SOURCE / "grid").glob("*.csv"))
stations_path = next(p for p in (SOURCE / "data").glob("*.csv")
                     if "마스터" in unicodedata.normalize("NFC", p.name))
boundary_path = SOURCE / "shp/bnd_dong_seoul.shp"
grid_path = SOURCE / "grid/seoul_100m_grid_2025.geojson"
zoning_path = next((SOURCE / "shp").glob("569_UQ111*")) / "UPIS_C_UQ111.shp"
dong = gpd.read_file(boundary_path).to_crs(CRS)
grid = gpd.read_file(grid_path).to_crs(CRS)
assert dong.geometry.is_valid.all() and grid.geometry.is_valid.all()
assert dong.ADM_CD.is_unique and grid.GRID_CD.is_unique

## 2. 총인구 추출과 격자 조인

총인구 코드 `to_in_001`만 선택하고 `GRID_CD`로 1:1 조인합니다. 원본 인구 파일은 서울보다 넓은 영역을 포함하므로 전체 합계를 서울 인구로 사용하지 않습니다.

In [ ]:
population = pd.read_csv(population_path, encoding="cp949", header=None,
    names=["year", "GRID_CD", "stat_code", "population"], dtype={"GRID_CD": str})
pop = population.loc[population.stat_code.eq("to_in_001")].copy()
assert pop.GRID_CD.is_unique and pop.year.eq(2024).all()
pop["population"] = pd.to_numeric(pop.population, errors="raise")
assert pop.population.ge(0).all()
grid = grid.merge(pop[["GRID_CD", "population"]], on="GRID_CD", how="left",
                  validate="one_to_one", indicator="population_match")
audit = {"grid_before_population_join": len(grid), "grid_after_population_join": len(grid),
         "to_in_001_rows_in_source_file": len(pop),
         "grids_with_missing_population": int(grid.population.isna().sum()),
         "matched_grid_population": float(grid.population.sum())}

## 3. 격자 중심점의 행정동 배정

중심점이 행정동 내부에 있는 격자를 배정합니다. 경계 밖 중심점과 공개 인구 미수록은 별도로 기록합니다. 미수록 격자가 실제로 무인구라는 뜻은 아닙니다.

In [ ]:
# 미수록 값을 실제 인구 0명으로 단정하지 않고 공개된 인구만 가중치에 반영한다.
grid["population_known"] = grid.population.notna()
grid["population"] = grid.population.fillna(0)
centers = grid.copy()
centers.geometry = centers.geometry.centroid
assigned = gpd.sjoin(centers, dong[["ADM_CD", "geometry"]], how="left", predicate="within")
assert assigned.GRID_CD.is_unique, "행정동 공간조인으로 격자가 중복되었습니다."
audit.update(grid_before_spatial_join=len(centers), grid_after_spatial_join=len(assigned),
             unassigned_grids=int(assigned.ADM_CD.isna().sum()),
             unassigned_population=float(assigned.loc[assigned.ADM_CD.isna(), "population"].sum()))
assigned.loc[assigned.ADM_CD.isna(), ["GRID_CD", "population"]].to_csv(
    OUT / "unassigned_grids.csv", index=False, encoding="utf-8-sig")
assigned = assigned.loc[assigned.ADM_CD.notna()].copy()

In [ ]:
display(pd.Series(audit, name="검증 기록").to_frame())

## 4. 역 좌표 정리와 최근접 거리

GTX는 제외합니다. 동일 좌표를 중복 제거하고 서울 경계 2km 이내 외부 역도 포함합니다. 호선이 다른 인접 좌표는 최근접역 계산에 중복 가산되지 않습니다.

In [ ]:
stations = pd.read_csv(stations_path, encoding="cp949")
audit["station_source_rows"] = len(stations)
# 이 연구는 일반 도시·광역철도 역의 지리적 접근성을 대상으로 한다.
# GTX는 요금·서비스 성격이 다르고 원본에 비영업 역도 있어 전부 제외한다.
audit["gtx_rows_excluded"] = int(stations.호선.eq("수도권 광역급행철도").sum())
stations = stations.loc[stations.호선.ne("수도권 광역급행철도")].copy()
assert stations[["경도", "위도"]].notna().all().all()
stations = stations.drop_duplicates(["경도", "위도"])
stations = gpd.GeoDataFrame(stations,
    geometry=gpd.points_from_xy(stations.경도, stations.위도), crs="EPSG:4326").to_crs(CRS)
boundary = dong.geometry.union_all()
# 절단거리 2km에 맞춘 외부 역 포함. 서울 경계에서의 과소평가를 줄인다.
stations = stations.loc[stations.geometry.intersects(boundary.buffer(2000))].copy()
audit["station_unique_coordinates_in_2km_buffer"] = len(stations)
# 원본은 호선마다 역 좌표를 따로 가진다. 개수는 역 수가 아니라 역·호선 좌표 점의 수이다.
stations["in_seoul"] = stations.geometry.intersects(boundary)
audit["station_points_inside_seoul"] = int(stations.in_seoul.sum())
tree = cKDTree(np.column_stack([stations.geometry.x, stations.geometry.y]))
distance, nearest = tree.query(np.column_stack([assigned.geometry.x, assigned.geometry.y]))
assigned["nearest_m"] = distance
assigned["equivalent_min"] = distance / (4000 / 60)
assigned["nearest_station"] = stations.iloc[nearest].역사명.to_numpy()

## 4-1. OSM 보행망 최단거리

격자 중심점과 역 좌표를 보행망의 가장 가까운 노드에 연결하고, 그 연결 거리를 더해 최근접역까지의 최단거리를 구합니다. 모든 역을 한꺼번에 출발점으로 두는 다중 출발 탐색이라 격자마다 최근접역까지의 거리가 한 번에 나옵니다. 4km를 넘으면 4km로 둡니다. 이후 접근성과 절대 접근 수준은 이 보행망 거리로 계산하고, 직선거리는 `line_m`에 남겨 비교합니다. 보행망 캐시(`.cache/seoul_walk_5179.pkl`)가 없을 때만 osmnx로 내려받습니다.

In [ ]:
import pickle
import networkx as nx

CACHE = ROOT / ".cache"
WALK_CACHE, WALK_META = CACHE / "seoul_walk_5179.pkl", CACHE / "seoul_walk_meta.json"
SPEED = 4000 / 60      # m/분 (시속 4km)
CUTOFF_M = 4000        # 보행망 탐색 상한(m). 넘으면 4,000m로 둔다.


def load_walk_graph():
    """서울 경계 2.5km 버퍼의 OSM 보행망(EPSG:5179, 무방향). 캐시가 없을 때만 내려받는다."""
    if WALK_CACHE.exists():
        return pickle.loads(WALK_CACHE.read_bytes()), json.loads(WALK_META.read_text())
    import osmnx as ox
    CACHE.mkdir(exist_ok=True)
    ox.settings.cache_folder = str(CACHE / "osmnx_http")
    ox.settings.requests_timeout = 600
    poly = dong.geometry.union_all().buffer(2500).simplify(50)
    poly_wgs = gpd.GeoSeries([poly], crs=CRS).to_crs(4326).iloc[0]
    graph = ox.graph_from_polygon(poly_wgs, network_type="walk", simplify=True)
    graph = ox.convert.to_undirected(ox.project_graph(graph, to_crs=CRS))
    meta = {"downloaded_kst": datetime.now(ZoneInfo("Asia/Seoul")).isoformat(timespec="seconds"),
            "osmnx": ox.__version__, "nodes": graph.number_of_nodes(), "edges": graph.number_of_edges(),
            "network_type": "walk", "buffer_m": 2500}
    WALK_CACHE.write_bytes(pickle.dumps(graph))
    WALK_META.write_text(json.dumps(meta, ensure_ascii=False, indent=2))
    return graph, meta


G_walk, walk_meta = load_walk_graph()
node_ids = np.array(list(G_walk.nodes), dtype=np.int64)
node_xy = np.array([(G_walk.nodes[k]["x"], G_walk.nodes[k]["y"]) for k in node_ids])
node_tree = cKDTree(node_xy)
grid_xy = np.column_stack([assigned.geometry.x, assigned.geometry.y])
station_xy = np.column_stack([stations.geometry.x, stations.geometry.y])
grid_snap, grid_idx = node_tree.query(grid_xy)
station_snap, station_idx = node_tree.query(station_xy)
grid_node, station_node = node_ids[grid_idx], node_ids[station_idx]


def walk_to_nearest(station_mask=None, snap=True):
    """선택한 역들에서 각 격자까지의 보행망 최단거리(m). 가상 출발점으로 다중 출발 탐색을 한다.
    snap=False이면 노드까지의 연결 거리를 더하지 않는다(민감도용)."""
    chosen = np.arange(len(stations)) if station_mask is None else np.flatnonzero(station_mask)
    source = "virtual_source"
    for k in chosen:
        G_walk.add_edge(source, int(station_node[k]), length=float(station_snap[k]) if snap else 0.0)
    try:
        reach = nx.single_source_dijkstra_path_length(G_walk, source, cutoff=CUTOFF_M, weight="length")
    finally:
        G_walk.remove_node(source)
    return np.array([reach.get(int(k), np.inf) for k in grid_node]) + (grid_snap if snap else 0.0)


total_m = walk_to_nearest()
line_m = assigned.nearest_m.to_numpy()
audit.update(walk_graph_nodes=int(G_walk.number_of_nodes()), walk_graph_edges=int(G_walk.number_of_edges()),
             walk_graph_downloaded_kst=walk_meta["downloaded_kst"],
             walk_over_4km_grids=int((total_m > CUTOFF_M).sum()),
             walk_shorter_than_line_by_50m_grids=int((total_m < line_m - 50).sum()))
audit.update(station_snap_median_m=float(np.median(station_snap)), station_snap_p90_m=float(np.percentile(station_snap, 90)),
             grid_snap_median_m=float(np.median(grid_snap)), grid_snap_p90_m=float(np.percentile(grid_snap, 90)))
assigned["line_m"] = line_m
assigned["walk_nosnap_m"] = np.minimum(walk_to_nearest(snap=False), CUTOFF_M)
assigned["walk_m"] = np.minimum(total_m, CUTOFF_M)
# 이후의 모든 지표는 보행망 거리(nearest_m)로 계산한다.
assigned["nearest_m"] = assigned.walk_m
assigned["equivalent_min"] = assigned.walk_m / SPEED
distance = assigned.nearest_m.to_numpy()
pd.DataFrame({"직선(m)": assigned.line_m.describe(), "보행망(m)": assigned.walk_m.describe()}).round(0)

## 5. 거리 감쇠와 인구가중 집계

격자 $i$의 공개 인구를 $P_i$, 행정동 면적을 $S_d$ (km²), 최근접역까지 OSM 보행망 최단거리를 $r_i$ (m)라 둡니다(직선거리는 비교용).

$$D_d=\frac{\sum_{i\in d}P_i}{S_d},\quad t_i=\frac{r_i}{4000/60},\quad\beta=\frac{\ln2}{10}$$
$$A_d=\frac{\sum_{i\in d}P_i e^{-\beta t_i}I(r_i\le2000)}{\sum_{i\in d}P_i}$$

접근성은 0~1이며 높을수록 공개 인구가 역 가까이에 분포합니다. $t_i$는 시속 4km로 환산한 값이며 실제 보행시간이 아닙니다. 이 지표는 최근접역 변형이며 모든 역 기여도의 합이 아닙니다.

보조 지표 (밀도 분모와 절대 수준을 점검하기 위한 값):

$$D^{built}_d=\frac{\sum_{i\in d}P_i}{S^{built}_d},\quad D^{res}_d=\frac{\sum_{i\in d}P_i^2}{\sum_{i\in d}P_i}\ (\text{명/ha}),\quad B_d=\frac{\sum_{i\in d}P_i I(r_i>1000)}{\sum_{i\in d}P_i}$$

$S^{built}_d$는 동 안의 주거·상업·공업지역 면적 합집합(km²), $D^{res}_d$는 100m 격자 한 칸(1ha)의 인구를 주민 수로 가중한 평균, $B_d$는 최근접역이 보행망 1km 밖인 인구 비율이다. 같은 값을 직선거리로 계산해 비교용으로 저장한다.

In [ ]:
for half in [5, 10, 15]:
    score = np.where(distance <= 2000, np.exp(-np.log(2) * assigned.equivalent_min / half), 0)
    assigned[f"weighted_{half}"] = assigned.population * score
assigned["weighted_distance"] = assigned.population * distance
assigned["population_500m"] = assigned.population.where(distance <= 500, 0)
assigned["population_beyond_1km"] = assigned.population.where(distance > 1000, 0)
assigned["population_sq"] = assigned.population ** 2
assigned["population_10min"] = assigned.population.where(distance <= SPEED * 10, 0)
aggregates = assigned.groupby("ADM_CD", as_index=False).agg(
    population=("population", "sum"), grids=("GRID_CD", "size"),
    known_grids=("population_known", "sum"), weighted_5=("weighted_5", "sum"),
    weighted_10=("weighted_10", "sum"), weighted_15=("weighted_15", "sum"),
    weighted_distance=("weighted_distance", "sum"), population_500m=("population_500m", "sum"),
    population_beyond_1km=("population_beyond_1km", "sum"), population_sq=("population_sq", "sum"),
    population_10min=("population_10min", "sum"))
result = dong.merge(aggregates, on="ADM_CD", how="left", validate="one_to_one")
assert len(result) == len(dong) and result.population.notna().all()
assert result.population.gt(0).all()
assert np.isclose(result.population.sum(), assigned.population.sum())
result["gu"] = result.ADM_CD.str[:5].map(GU)
assert result.gu.notna().all()
result["name"] = result.gu + " " + result.ADM_NM
result["area_km2"] = result.area / 1e6
result["density"] = result.population / result.area_km2
for half in [5, 10, 15]:
    result[f"access_{half}"] = result[f"weighted_{half}"] / result.population
    assert result[f"access_{half}"].between(0, 1).all()
result["access"] = result.access_10
result["mean_distance_m"] = result.weighted_distance / result.population
result["within_500m_pct"] = result.population_500m / result.population * 100
result["beyond_1km_pct"] = result.population_beyond_1km / result.population * 100
# 100m 격자 한 칸은 1ha이다. 주민이 실제로 사는 격자의 밀도를 주민 수로 가중한 값(명/ha).
result["resident_density_ha"] = result.population_sq / result.population
result["beyond_1km_pop"] = result.population_beyond_1km
result["within_10min_pct"] = result.population_10min / result.population * 100
# 비교용: 같은 지표를 직선거리로 계산한다.
line = pd.DataFrame({
    "w": assigned.population * np.where(assigned.line_m <= 2000, np.exp(-np.log(2) * (assigned.line_m / SPEED) / 10), 0),
    "d": assigned.population * assigned.line_m,
    "p500": assigned.population.where(assigned.line_m <= 500, 0),
    "p1k": assigned.population.where(assigned.line_m > 1000, 0)}).groupby(assigned.ADM_CD).sum()
pop_by = aggregates.set_index("ADM_CD").population
result["access_line"] = result.ADM_CD.map(line.w / pop_by)
result["mean_distance_line_m"] = result.ADM_CD.map(line.d / pop_by)
result["within_500m_line_pct"] = result.ADM_CD.map(line.p500 / pop_by * 100)
result["beyond_1km_line_pct"] = result.ADM_CD.map(line.p1k / pop_by * 100)

## 5-1. 용도지역 기반 보조 밀도

동 전체 면적에는 산지와 하천이 들어 있어 밀도가 낮게 나올 수 있습니다. 주거·상업·공업지역(`UQA100`·`UQA200`·`UQA300`)의 합집합 면적을 분모로 쓴 밀도를 보조로 계산합니다. 용도지역 원본에는 같은 부류의 중복 다각형이 있어(합 766km² 대비 합집합 605km²) 면적을 합집합으로 계산합니다. 이 파일은 파일명상 2026-09 기준이라 2025 행정동 경계와 시점이 다릅니다.

In [ ]:
zones = gpd.read_file(zoning_path).to_crs(CRS)
zones["geometry"] = zones.geometry.make_valid()
built = zones.loc[zones.LCLAS_CL.isin(["UQA100", "UQA200", "UQA300"])]
built_union = built.geometry.union_all()
audit["zoning_polygons"] = len(zones)
audit["zoning_built_polygons"] = len(built)
result["built_km2"] = result.geometry.intersection(built_union).area / 1e6
result["built_share"] = result.built_km2 / result.area_km2
# 지역 지정 면적이 없는 동은 NaN으로 두고 분석에서 제외한다.
result["density_built"] = result.population / result.built_km2.where(result.built_km2 > 0)
audit["dong_without_built_zone"] = int(result.density_built.isna().sum())
audit["built_km2_total"] = float(result.built_km2.sum())
result[["built_share", "density", "density_built", "resident_density_ha"]].describe().round(2)

## 5-2. 생활인구 보조 변수

수업 7주차 생활인구(14시·03시)를 행정동에 붙여 밀도 점검에만 씁니다. 행정동 코드 체계가 달라 자치구명과 동명으로 연결하고, 가운뎃점(·)은 마침표로 통일합니다. `SPOP_mid`는 수업 6주차 `02_living_population_time_series_visualization`에서 2025년 일별 시간대별 생활인구(서울 열린데이터광장 OA-23016, 수업 `01_living_population_data_preparation`이 월별 zip을 내려받아 합침)를 행정동·시간대별로 산술평균한 값입니다. 요일·공휴일을 구분하지 않았고, 수업 노트북은 원자료에 결측이 많다고 적어 두었습니다.

In [ ]:
spop_path = SOURCE.parent / "lecture06_Living_Population/data/SPOP_2025_ADM_mid.csv"
spop = pd.read_csv(spop_path, dtype={"TT_str": str})
spop = (spop.loc[spop.TT_str.isin(["03", "14"])]
        .pivot_table(index=["SGG_NM", "ADMI_NM"], columns="TT_str", values="SPOP_mid").reset_index())
spop.columns = ["gu", "dong_key", "lp_03", "lp_14"]
lp = pd.DataFrame({"gu": result.gu.to_numpy(), "dong_key": result.ADM_NM.str.replace("·", ".", regex=False).to_numpy()}
                  ).merge(spop, on=["gu", "dong_key"], how="left", validate="one_to_one")
result["lp_03"], result["lp_14"] = lp.lp_03.to_numpy(), lp.lp_14.to_numpy()
audit["living_pop_matched_dongs"] = int(result.lp_14.notna().sum())
assert audit["living_pop_matched_dongs"] == len(result), "생활인구와 연결되지 않은 동이 있습니다."
result["lp14_density"] = result.lp_14 / result.area_km2
result["lp_day_over_resident"] = result.lp_14 / result.population
result[["lp_03", "lp_14", "lp14_density", "lp_day_over_resident"]].describe().round(2)

## 6. 후보 선별·상관관계·민감도

밀도 상위 25%와 접근성 하위 25%를 동시에 만족하면 추가 검토 후보입니다. 공식 취약지역 판정이 아닙니다. 두 조건이 겹치는 동의 수가 서로 독립일 때 기대되는 수보다 많은지 순열 검정(10,000회)으로 확인합니다. 민감도는 반감기, 분위 기준, 역 집합, 모든 역 합산, 밀도 분모를 바꿔 후보가 얼마나 유지되는지 봅니다. 동별 상관은 인과관계나 개인 수준의 교통 경험을 설명하지 않습니다.

In [ ]:
SHARE = .25
FAR_PCT = 50   # 인구의 절반 이상이 최근접역 1km 밖인 동
BETA_MIN = 4000 / 60


def pick(density, access, share=SHARE):
    """밀도 상위 share와 접근성 하위 share를 동시에 만족하는 동. NaN은 제외된다."""
    d = pd.Series(np.asarray(density, dtype=float), index=result.index)
    a = pd.Series(np.asarray(access, dtype=float), index=result.index)
    return d.ge(d.quantile(1 - share)) & a.le(a.quantile(share))


high = float(result.density.quantile(1 - SHARE))
low = float(result.access.quantile(SHARE))
result["priority"] = pick(result.density, result.access)
result["built_priority"] = pick(result.density_built, result.access)
result["type"] = np.select([
    result.priority, result.density.ge(high), result.access.le(low)],
    ["고밀도·저접근성", "고밀도", "저접근성"], default="그 외")
sp = spearmanr(result.density, result.access)
pe = pearsonr(result.density, result.access)
rho, rho_p = float(sp.statistic), float(sp.pvalue)
r, r_p = float(pe.statistic), float(pe.pvalue)
med_density, med_access = float(result.density.median()), float(result.access.median())
candidates = result.loc[result.priority].sort_values("density", ascending=False)

# 겹침이 우연보다 많은가: 접근성 값을 동 사이에서 무작위로 섞어 비교한다.
is_high = result.density.ge(high).to_numpy()
is_low = result.access.le(low).to_numpy()
overlap = {"high_count": int(is_high.sum()), "low_count": int(is_low.sum()),
           "observed": int((is_high & is_low).sum()),
           "expected_if_independent": float(is_high.sum() * is_low.sum() / len(result))}
rng = np.random.default_rng(20261101)
access_values = result.access.to_numpy()
sims = np.array([(is_high & (rng.permutation(access_values) <= low)).sum() for _ in range(10000)])
overlap["permutation_p_at_least_observed"] = float((sims >= overlap["observed"]).mean())

# 서울 전체의 절대 접근 수준과 후보 동의 절대 수준
w, grid_dist = assigned.population.to_numpy(), distance
citywide = {"weighted_mean_distance_m": float((w * grid_dist).sum() / w.sum()),
            "within_500m_pct": float(w[grid_dist <= 500].sum() / w.sum() * 100),
            "within_1000m_pct": float(w[grid_dist <= 1000].sum() / w.sum() * 100),
            "beyond_2000m_pct": float(w[grid_dist > 2000].sum() / w.sum() * 100)}
candidate_absolute = {"median_mean_distance_m": float(candidates.mean_distance_m.median()),
                      "median_within_500m_pct": float(candidates.within_500m_pct.median()),
                      "median_beyond_1km_pct": float(candidates.beyond_1km_pct.median()),
                      "max_mean_distance_m": float(candidates.mean_distance_m.max())}
citywide.update(beyond_1km_pct=float(w[grid_dist > 1000].sum() / w.sum() * 100),
                beyond_1km_people=float(w[grid_dist > 1000].sum()),
                within_10min_pct=float(w[grid_dist <= SPEED * 10].sum() / w.sum() * 100),
                unreachable_4km_pct=float(w[grid_dist >= CUTOFF_M].sum() / w.sum() * 100),
                capped_4km_people=float(w[grid_dist >= CUTOFF_M].sum()))
wl = assigned.line_m.to_numpy()
citywide_line = {"weighted_mean_distance_m": float((w * wl).sum() / w.sum()),
                 "within_500m_pct": float(w[wl <= 500].sum() / w.sum() * 100),
                 "within_1000m_pct": float(w[wl <= 1000].sum() / w.sum() * 100),
                 "beyond_1km_pct": float(w[wl > 1000].sum() / w.sum() * 100),
                 "beyond_1km_people": float(w[wl > 1000].sum())}
network_vs_line = {"mean_distance_ratio": citywide["weighted_mean_distance_m"] / citywide_line["weighted_mean_distance_m"],
                   "spearman_dong_access": float(spearmanr(result.access, result.access_line).statistic),
                   "spearman_dong_mean_distance": float(spearmanr(result.mean_distance_m, result.mean_distance_line_m).statistic),
                   "far_dongs_walk": int(result.beyond_1km_pct.ge(FAR_PCT).sum()),
                   "far_dongs_line": int(result.beyond_1km_line_pct.ge(FAR_PCT).sum()),
                   "nosnap_beyond_1km_pct": float(w[assigned.walk_nosnap_m.to_numpy() > 1000].sum() / w.sum() * 100),
                   "nosnap_beyond_1km_people": float(w[assigned.walk_nosnap_m.to_numpy() > 1000].sum())}
# 토지이용: 주거·상업·공업지역 비율(= 1 - 녹지·미지정 비율)과 두 지표의 관계
landuse = {"spearman_built_share_access": float(spearmanr(result.built_share, result.access).statistic),
           "spearman_built_share_density": float(spearmanr(result.built_share, result.density).statistic),
           "median_built_share_candidates": float(candidates.built_share.median()),
           "median_built_share_all": float(result.built_share.median())}

# ---- 민감도: 반감기, 분위 기준, 역 집합, 모든 역 합산, 밀도 분모
pop_w, codes = assigned.population.to_numpy(), assigned.ADM_CD.to_numpy()
pop_by_dong = pd.Series(pop_w).groupby(codes).sum()


def grid_score(dist_m, half=10):
    return np.where(dist_m <= 2000, np.exp(-np.log(2) * (dist_m / BETA_MIN) / half), 0)


def dong_mean(values):
    mean = pd.Series(pop_w * values).groupby(codes).sum() / pop_by_dong
    return mean.reindex(result.ADM_CD).to_numpy()


inside = stations.in_seoul.to_numpy()
dist_inside = np.minimum(walk_to_nearest(inside), CUTOFF_M)
node_rows = {}
for row_idx, node in enumerate(grid_node):
    node_rows.setdefault(int(node), []).append(row_idx)
node_rows = {k: np.asarray(v) for k, v in node_rows.items()}


def all_station_score(weights):
    """모든 역의 기여를 합산한 보행망 중력형 점수(격자별). 보행망 2km 안의 역만 센다."""
    score = np.zeros(len(grid_node))
    for k in range(len(stations)):
        reach = nx.single_source_dijkstra_path_length(G_walk, int(station_node[k]), cutoff=2000, weight="length")
        for node, d in reach.items():
            rows = node_rows.get(node)
            if rows is None:
                continue
            walk = d + station_snap[k] + grid_snap[rows]
            ok = walk <= 2000
            score[rows[ok]] += weights[k] * np.exp(-np.log(2) * (walk[ok] / SPEED) / 10)
    return score


name_count = stations.groupby("역사명")["역사명"].transform("size").to_numpy()
all_points = all_station_score(np.ones(len(stations)))
all_by_name = all_station_score(1 / name_count)

scenarios = [
    ("기준: 보행망, 반감기 10분, 상·하위 25%", result.density, result.access, .25),
    ("반감기 5분", result.density, result.access_5, .25),
    ("반감기 15분", result.density, result.access_15, .25),
    ("상·하위 20% 기준", result.density, result.access, .20),
    ("상·하위 30% 기준", result.density, result.access, .30),
    ("직선거리 사용", result.density, result.access_line, .25),
    ("연결 거리 제외 (노드 간 거리만)", result.density, dong_mean(grid_score(assigned.walk_nosnap_m.to_numpy())), .25),
    ("서울 안 역만 사용", result.density, dong_mean(grid_score(dist_inside)), .25),
    ("모든 역 합산 (점 단위 가산)", result.density, dong_mean(all_points), .25),
    ("모든 역 합산 (역사명 보정)", result.density, dong_mean(all_by_name), .25),
    ("주거·상업·공업지역 면적 밀도", result.density_built, result.access, .25),
    ("주민가중 격자 밀도", result.resident_density_ha, result.access, .25),
    ("생활인구(14시) 밀도", result.lp14_density, result.access, .25),
]
base_ids = set(candidates.ADM_CD)
hits = pd.Series(0, index=result.ADM_CD)
chosen = {}
sensitivity = []
for label, dens, acc, share in scenarios:
    ids = set(result.loc[pick(dens, acc, share), "ADM_CD"])
    hits.loc[list(ids)] += 1
    chosen[label] = ids
    sensitivity.append({"scenario": label, "candidate_count": len(ids),
        "shared_with_baseline": len(base_ids & ids),
        "spearman_density_access": float(spearmanr(np.asarray(dens, float), np.asarray(acc, float), nan_policy="omit").statistic)})
result["scenario_hits"] = result.ADM_CD.map(hits)
candidates = result.loc[result.priority].sort_values("density", ascending=False)
robust = candidates.loc[candidates.scenario_hits.eq(len(scenarios)), "name"].tolist()
name_of = result.set_index("ADM_CD").name
robust_density = name_of.loc[sorted(base_ids & chosen["주거·상업·공업지역 면적 밀도"] & chosen["주민가중 격자 밀도"])].tolist()
strong = candidates.loc[candidates.beyond_1km_pct.ge(FAR_PCT), "name"].tolist()
far = result.loc[result.beyond_1km_pct.ge(FAR_PCT)].sort_values("beyond_1km_pop", ascending=False)
top_beyond = result.nlargest(15, "beyond_1km_pop")
living = {"spearman_lp14_density_access": float(spearmanr(result.lp14_density, result.access).statistic),
          "spearman_lp14_density_density": float(spearmanr(result.lp14_density, result.density).statistic),
          "share_day_over_resident": float(result.lp_day_over_resident.gt(1).mean())}
overlap["low_access_with_density_below_median"] = int((is_low & result.density.le(med_density).to_numpy()).sum())

# ---- 자치구별 요약
gu_table = result.groupby("gu").agg(dongs=("ADM_CD", "size"), population=("population", "sum"),
    area_km2=("area_km2", "sum"), weighted_10=("weighted_10", "sum"), population_500m=("population_500m", "sum"),
    population_beyond_1km=("population_beyond_1km", "sum"), candidates=("priority", "sum")).reset_index()
gu_table["density"] = gu_table.population / gu_table.area_km2
gu_table["access"] = gu_table.weighted_10 / gu_table.population
gu_table["within_500m_pct"] = gu_table.population_500m / gu_table.population * 100
gu_table["beyond_1km_pct"] = gu_table.population_beyond_1km / gu_table.population * 100
gu_table = gu_table.drop(columns=["weighted_10", "population_500m", "population_beyond_1km"]).sort_values("access").reset_index(drop=True)


In [ ]:
display(candidates[["name", "density", "access", "mean_distance_m", "beyond_1km_pct", "built_share", "scenario_hits"]].round(3))
display(pd.DataFrame(sensitivity).round(3))
print(f"Pearson r={r:.3f} (p={r_p:.3f}), Spearman ρ={rho:.3f} (p={rho_p:.3f})")
print(f"겹침 관측 {overlap['observed']}개, 독립일 때 기대 {overlap['expected_if_independent']:.1f}개, "
      f"기대 이상일 확률 p={overlap['permutation_p_at_least_observed']:.3f}")
print("서울 전체:", {k: round(v, 1) for k, v in citywide.items()})
print("후보 동:", {k: round(v, 1) for k, v in candidate_absolute.items()})
print("토지이용:", {k: round(v, 3) for k, v in landuse.items()})
print("모든 시나리오에서 유지된 후보:", robust)
print("두 밀도 분모에서 모두 유지된 후보:", robust_density)
print("인구 절반 이상이 1km 밖인 후보:", strong)
display(gu_table.round(3))
print("보행망:", {k: round(v, 2) for k, v in citywide.items()})
print("직선:", {k: round(v, 2) for k, v in citywide_line.items()})
print("보행망 vs 직선:", {k: round(v, 3) for k, v in network_vs_line.items()})
print("생활인구:", {k: round(v, 3) for k, v in living.items()})
print("1km 밖 인구 절반 이상인 동:", len(far), "개")
display(top_beyond[["name", "beyond_1km_pop", "beyond_1km_pct", "density", "access"]].round(2))

## 7. 결과와 출처 검증 기록 저장

조인 행 수, 누락 인구, 원본 파일 해시, 집계 결과를 저장합니다. 웹 앱에는 간소화 경계와 집계값만 전달합니다.

In [ ]:
timestamp = datetime.now(ZoneInfo("Asia/Seoul")).isoformat(timespec="seconds")
summary = {"analysis_time_kst": timestamp, "dong_count": len(result),
    "public_population": int(result.population.sum()), "median_density": med_density,
    "median_access": med_access, "density_q75": high, "access_q25": low,
    "candidate_count": len(candidates), "pearson_r": r, "pearson_p": r_p,
    "spearman_rho": rho, "spearman_p": rho_p, "overlap": overlap, "citywide": citywide,
    "candidate_absolute": candidate_absolute, "landuse": landuse, "scenario_count": len(scenarios),
    "citywide_line": citywide_line, "network_vs_line": network_vs_line, "living": living,
    "far_threshold_pct": FAR_PCT,
    "far_dongs": json.loads(far[["name", "beyond_1km_pop", "beyond_1km_pct", "population", "density", "access"]]
                            .round(3).to_json(orient="records", force_ascii=False)),
    "top_beyond": json.loads(top_beyond[["name", "beyond_1km_pop", "beyond_1km_pct", "population", "density", "access"]]
                             .round(3).to_json(orient="records", force_ascii=False)),
    "robust_all": robust, "robust_density": robust_density, "strong_candidates": strong,
    "sensitivity": sensitivity, "gu": json.loads(gu_table.round(4).to_json(orient="records", force_ascii=False)),
    "audit": audit}
columns = ["ADM_CD", "name", "gu", "ADM_NM", "population", "area_km2", "density",
           "access", "mean_distance_m", "within_500m_pct", "beyond_1km_pct", "built_share", "density_built",
           "resident_density_ha", "scenario_hits", "beyond_1km_pop", "within_10min_pct", "access_line",
           "mean_distance_line_m", "within_500m_line_pct", "beyond_1km_line_pct", "lp_03", "lp_14",
           "lp14_density", "built_priority", "type", "priority", "grids", "known_grids"]
result[columns].to_csv(OUT / "dong_indicators.csv", index=False, encoding="utf-8-sig")
candidates[columns].to_csv(OUT / "priority_areas.csv", index=False, encoding="utf-8-sig")
gu_table.to_csv(OUT / "gu_summary.csv", index=False, encoding="utf-8-sig")
result[columns + ["geometry"]].to_crs(4326).to_file(OUT / "dong_indicators.geojson", driver="GeoJSON")
(OUT / "summary.json").write_text(json.dumps(summary, ensure_ascii=False, indent=2))
assigned[["GRID_CD", "ADM_CD", "population", "population_known", "nearest_m",
          "nearest_station", "equivalent_min", "walk_m", "line_m"]].to_csv(OUT / "grid_results.csv", index=False, encoding="utf-8-sig")
manifest = []
BASE = ROOT.parents[1]
for p, when in [(population_path, "확인 못함"), (grid_path, "확인 못함"), (stations_path, "확인 못함"),
                (zoning_path, "확인 못함"), (zoning_path.with_suffix(".dbf"), "확인 못함"), (spop_path, "확인 못함"),
                *[(q, "확인 못함") for q in sorted((SOURCE / "shp").glob("bnd_dong_seoul.*"))],
                (WALK_CACHE, walk_meta["downloaded_kst"])]:
    manifest.append({"path": str(p.relative_to(BASE)), "sha256": hashlib.sha256(p.read_bytes()).hexdigest(),
                     "bytes": p.stat().st_size, "original_collection_date": when})
(OUT / "source_manifest.json").write_text(json.dumps(manifest, ensure_ascii=False, indent=2))
# 브라우저에는 간소화한 경계와 집계 결과만 전달한다. 원본 격자는 포함하지 않는다.
web = result[columns + ["geometry"]].copy()
web.geometry = web.geometry.simplify(30, preserve_topology=True)
map_data = json.loads(web.to_json(drop_id=True))
(DIST / "data.js").write_text("window.STUDY=" + json.dumps({"summary": summary, "geo": map_data}, ensure_ascii=False) + ";\n")
for file in ["dong_indicators.csv", "priority_areas.csv", "gu_summary.csv", "source_manifest.json", "summary.json"]:
    (DIST / file).write_bytes((OUT / file).read_bytes())

## 7-1. 세 지역의 확대 사례

같은 후보 안의 절대 거리 차이와 밀도 분모에 따른 후보 변화를 비교합니다. 사례는 규칙으로 고릅니다. 첫째는 후보 중 1km 밖 인구 비율이 가장 낮은 동, 둘째는 가장 높은 동, 셋째는 전체 면적 밀도로는 후보가 아니지만 개발지역 면적 밀도로는 후보가 되는 동 가운데 개발지역 비율이 가장 낮은 동입니다. **원래 지표와 후보 기준은 유지합니다.**

역을 중심으로 그린 원은 기준 역에서 직선 500m·1km 범위입니다. 기준 역은 직선 최근접역으로 이 동의 공개 인구가 가장 많이 배정된 역이라 보행망 최근접역과 다를 수 있습니다. 주황 선은 동 평균 거리에 가까운 격자 하나의 보행망 최단 경로입니다. 인구 미수록은 실제 무인구로 단정하지 않습니다. 개발지역은 용도지역 분류상 주거·상업·공업지역의 합집합이며 실제 건물이 있는 면적은 아닙니다.

In [ ]:
# 기존 집계 결과와 원본 격자에서 사례 3곳만 추출한다. 외부 API는 호출하지 않는다.
from shapely.geometry import mapping

cand_pool = result.loc[result.priority]
flip_pool = result.loc[~result.priority & result.built_priority]
if flip_pool.empty:
    flip_pool = result.loc[~result.priority].nsmallest(1, "built_share")
CASE_NAMES = [
    cand_pool.sort_values(["beyond_1km_pct", "density"], ascending=[True, False]).name.iloc[0],
    cand_pool.sort_values("beyond_1km_pct", ascending=False).name.iloc[0],
    flip_pool.sort_values("built_share").name.iloc[0],
]
def example_route(case_points, row):
    """동의 평균 거리에 가까운 격자 하나를 골라 보행망 최단 경로를 구한다(모든 역을 가상 출발점에 묶어 한 번에 탐색)."""
    pool = case_points.loc[(case_points.population > 0) & (case_points.nearest_m < CUTOFF_M)]
    chosen = (pool.assign(gap=(pool.nearest_m - row.mean_distance_m).abs())
              .nsmallest(5, "gap").nlargest(1, "population").iloc[0])
    k = int(np.flatnonzero(assigned.GRID_CD.to_numpy() == chosen.GRID_CD)[0])
    source = "virtual_route"
    for j in range(len(stations)):
        G_walk.add_edge(source, int(station_node[j]), length=float(station_snap[j]))
    try:
        length, path = nx.single_source_dijkstra(G_walk, source, target=int(grid_node[k]), weight="length")
    finally:
        G_walk.remove_node(source)
    j = min((j for j in range(len(stations)) if int(station_node[j]) == path[1]), key=lambda j: station_snap[j])
    coords = ([[float(station_xy[j][0]), float(station_xy[j][1])]]
              + [[float(G_walk.nodes[v]["x"]), float(G_walk.nodes[v]["y"])] for v in path[1:]]
              + [[float(grid_xy[k][0]), float(grid_xy[k][1])]])
    parts = {"grid_snap_m": float(grid_snap[k]), "network_m": float(length - station_snap[j]),
             "station_snap_m": float(station_snap[j]), "walk_m": float(length + grid_snap[k]),
             "line_m": float(assigned.line_m.iloc[k])}
    assert abs(parts["walk_m"] - float(assigned.walk_m.iloc[k])) < .01, "예시 경로 길이가 격자 결과와 다릅니다."
    st_row = stations.iloc[j]
    return {"grid_code": chosen.GRID_CD, "population": float(chosen.population), "coords": coords, "parts": parts,
            "station_name": st_row.역사명, "station_line": st_row.호선}


case_maps = []
case_plot_layers = []
for case_name in CASE_NAMES:
    row = result.loc[result.name.eq(case_name)].iloc[0]
    case_points = assigned.loc[assigned.ADM_CD.eq(row.ADM_CD)].copy()
    case_grids = grid.loc[grid.GRID_CD.isin(case_points.GRID_CD)].merge(
        case_points[["GRID_CD", "nearest_m", "line_m"]], on="GRID_CD", validate="one_to_one")
    assert np.isclose(case_grids.population.sum(), row.population)
    station_index = tree.query(np.column_stack([case_grids.geometry.centroid.x,
                                              case_grids.geometry.centroid.y]))[1]
    case_grids["nearest_station_key"] = station_index.astype(int)
    # 동의 공개 인구가 최근접역으로 가장 많이 배정된 좌표를 기본 선택한다.
    default_key = int(pd.Series(case_grids.population.to_numpy()).groupby(station_index).sum().idxmax())
    extent = row.geometry.buffer(1200).union(stations.iloc[default_key].geometry.buffer(1050)).envelope
    context = result.loc[result.geometry.intersects(extent), ["ADM_CD", "ADM_NM", "geometry"]].copy()
    context.geometry = context.geometry.intersection(extent).simplify(8)
    local_stations = stations.loc[stations.geometry.intersects(extent)].copy()
    # 선택 동 주민의 최근접역이 범위 바깥에 있더라도 조회할 수 있게 포함한다.
    keys = sorted(set(local_stations.index) | set(stations.iloc[np.unique(station_index)].index))
    local_stations = stations.loc[keys]
    local_station_keys = [int(stations.index.get_loc(i)) for i in local_stations.index]
    features = []
    for _, g in case_grids.iterrows():
        features.append({"type": "Feature", "geometry": mapping(g.geometry), "properties": {
            "code": g.GRID_CD, "population": float(g.population),
            "known": bool(g.population_known), "distance": float(g.nearest_m), "line": float(g.line_m),
            "station_key": int(g.nearest_station_key)}})
    route = example_route(case_points, row)
    record = {"name": case_name, "id": row.ADM_CD, "route": route,
        "metrics": {k: float(row[k]) for k in ["density", "density_built", "access", "population",
                    "mean_distance_m", "mean_distance_line_m", "within_500m_pct", "beyond_1km_pct",
                    "beyond_1km_line_pct", "built_share"]},
        "priority": bool(row.priority), "built_priority": bool(row.built_priority),
        "boundary": mapping(row.geometry), "context": json.loads(context.to_json(drop_id=True)),
        "built": mapping(built_union.intersection(extent)), "grids": features,
        "stations": [{"key": key, "name": st.역사명, "line": st.호선,
                      "x": float(st.geometry.x), "y": float(st.geometry.y)}
                     for key, (_, st) in zip(local_station_keys, local_stations.iterrows())],
        "default_station": default_key, "bounds": list(extent.bounds)}
    assert record["default_station"] in [st["key"] for st in record["stations"]]
    case_maps.append(record)
    case_plot_layers.append((row, case_grids, context, local_stations,
                             stations.iloc[default_key], built_union.intersection(extent), extent))

case_payload = {"crs": CRS, "density_built_q75": float(result.density_built.quantile(.75)),
    "cases": case_maps,
    "stations": [{"name": st.역사명, "line": st.호선,
                  "x": float(st.geometry.x), "y": float(st.geometry.y)} for _, st in stations.iterrows()]}
(DIST / "cases.js").write_text("window.CASE_STUDY=" + json.dumps(case_payload, ensure_ascii=False) + ";\n")
(OUT / "case_summary.json").write_text(json.dumps(
    [{k: c[k] for k in ["name", "metrics", "priority", "built_priority"]} for c in case_maps],
    ensure_ascii=False, indent=2))
display(pd.DataFrame([{ "지역": c["name"], "인구밀도": c["metrics"]["density"],
    "개발지역밀도": c["metrics"]["density_built"], "접근성": c["metrics"]["access"],
    "1km밖인구비율(보행망)": c["metrics"]["beyond_1km_pct"], "기존후보": c["priority"],
    "개발지역밀도후보": c["built_priority"]} for c in case_maps]).round(3))

# 노트북과 보고서에서도 같은 사례를 확인할 수 있는 지도 파일을 저장한다.
os.environ["MPLCONFIGDIR"] = str(OUT / ".matplotlib")
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap, BoundaryNorm
from matplotlib.patches import Circle, Patch
from matplotlib import font_manager
installed = {f.name for f in font_manager.fontManager.ttflist}
plt.rcParams["font.family"] = next((f for f in ["AppleGothic", "Malgun Gothic", "NanumGothic"] if f in installed), "sans-serif")
colors = ["#f6dfa0", "#dbad55", "#b87932", "#784b24"]
fig, axes = plt.subplots(1, 3, figsize=(17, 6))
for ax, (row, cg, context, st, default_station, built_geom, extent), case in zip(axes, case_plot_layers, case_maps):
    context.boundary.plot(ax=ax, color="#cdd4cf", linewidth=.6)
    gpd.GeoSeries([built_geom], crs=CRS).plot(ax=ax, color="#dcebe0", alpha=.65)
    cg.loc[~cg.population_known].plot(ax=ax, color="#dddddd", linewidth=0)
    inhabited = cg.loc[cg.population > 0]
    inhabited.plot(ax=ax, column="population", cmap=ListedColormap(colors),
                   norm=BoundaryNorm([1, 100, 300, 600, 100000], 4), linewidth=.15, edgecolor="white")
    gpd.GeoSeries([row.geometry], crs=CRS).boundary.plot(ax=ax, color="#18342e", linewidth=1.5)
    for radius in [500, 1000]:
        ax.add_patch(Circle((default_station.geometry.x, default_station.geometry.y), radius,
                           fill=False, edgecolor="#2783a2", linewidth=1, linestyle="--"))
    st.plot(ax=ax, color="#18342e", markersize=14)
    rx, ry = zip(*case["route"]["coords"])
    ax.plot(rx, ry, color="#d94801", linewidth=2.2, zorder=6)
    ax.scatter(rx[-1:], ry[-1:], color="#d94801", s=36, zorder=7)
    ax.annotate(default_station.역사명, (default_station.geometry.x, default_station.geometry.y),
                xytext=(4, 5), textcoords="offset points", fontsize=9)
    x0, y0, x1, y1 = extent.bounds
    ax.set(xlim=(x0, x1), ylim=(y0, y1), title=f"{row['name']}\n보행망 평균 {row.mean_distance_m:,.0f}m(직선 {row.mean_distance_line_m:,.0f}m) · 1km 밖 {row.beyond_1km_pct:.1f}%")
    ax.set_axis_off(); ax.set_aspect("equal")
fig.legend(handles=[Patch(color=c, label=l) for c, l in zip(colors,
    ["1~99명", "100~299명", "300~599명", "600명 이상"])], loc="lower center", ncol=4)
fig.suptitle("동 평균 뒤에 있는 인구 격자와 역의 배치", fontsize=17)
fig.text(.5, .06, "연녹색: 주거·상업·공업지역 / 회색 격자: 인구 미수록 / 점선: 직선 최근접역 기준 500m·1km 직선 범위 / 주황 선: 예시 격자의 보행망 최단 경로", ha="center", fontsize=10)
fig.tight_layout(rect=(0, .10, 1, .94))
fig.savefig(OUT / "case_maps.png", dpi=180, bbox_inches="tight")
plt.close(fig)
display(Image(filename=str(OUT / "case_maps.png")))


## 8. 지도·산점도·자치구 비교

붉은 경계·점은 고밀도·저접근성 후보입니다. 접근성 지도의 검은 점은 서울 안 역·호선 좌표입니다. 자치구 그림은 구별 인구가중 접근성과 구 안 행정동의 분포를 비교합니다.

In [ ]:
def figures(result, stations, gu_table):
    os.environ["MPLCONFIGDIR"] = str(OUT / ".matplotlib")
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    from matplotlib import font_manager
    installed = {f.name for f in font_manager.fontManager.ttflist}
    plt.rcParams["font.family"] = next((f for f in ["AppleGothic", "Malgun Gothic", "NanumGothic"] if f in installed), "sans-serif")
    plt.rcParams["axes.unicode_minus"] = False
    fig, axes = plt.subplots(1, 2, figsize=(14, 7))
    for ax, column, title, cmap in zip(axes, ["density", "access"],
        ["인구밀도 (명/km²)", "최근접역 중력형 접근성 (0~1)"], ["YlOrBr", "GnBu"]):
        result.plot(column=column, cmap=cmap, ax=ax, legend=True, edgecolor="white", linewidth=.15)
        result.loc[result.priority].boundary.plot(ax=ax, color="#c34645", linewidth=.9)
        if column == "access":
            stations.loc[stations.in_seoul].plot(ax=ax, color="#1b1b1b", markersize=3, zorder=5)
        ax.set_title(title); ax.set_axis_off()
    fig.suptitle("사람이 많이 사는 곳은 지하철도 가까울까?", fontsize=17)
    fig.text(.5, .02, "2024 공개 격자 인구 · 2025 행정동 | 보행망 거리 기준 · GTX 제외 | 붉은 경계: 고밀도·저접근성 후보", ha="center", fontsize=10)
    fig.savefig(OUT / "overview.png", dpi=180, bbox_inches="tight"); plt.close(fig)
    fig, ax = plt.subplots(figsize=(10, 7))
    ax.scatter(result.density, result.access, c=np.where(result.priority, "#c34645", "#24858d"), alpha=.65, s=28)
    ax.axvline(result.density.quantile(.75), color="#777777", linestyle="--")
    ax.axhline(result.access.quantile(.25), color="#777777", linestyle="--")
    for _, row in result.loc[result.priority].nlargest(5, "density").iterrows():
        ax.annotate(row.ADM_NM, (row.density, row.access), xytext=(4, 4), textcoords="offset points", fontsize=9)
    ax.set(xlabel="행정동 전체 면적 기준 인구밀도 (명/km²)", ylabel="최근접역 중력형 접근성 (0~1)",
           title="고밀도 지역의 주민은 역 가까이에 살고 있을까?")
    fig.tight_layout(); fig.savefig(OUT / "scatter.png", dpi=180); plt.close(fig)
    order = gu_table.sort_values("access").reset_index(drop=True)
    ypos = np.arange(len(order))
    fig, axes = plt.subplots(1, 2, figsize=(14, 8), sharey=True)
    axes[0].barh(ypos, order.access, color="#24858d")
    for y, (acc, cand) in enumerate(zip(order.access, order.candidates)):
        axes[0].text(acc + .01, y, f"{acc:.2f} · 후보 {int(cand)}개", va="center", fontsize=9)
    axes[0].set_yticks(ypos, order.gu); axes[0].set_xlim(0, 1)
    axes[0].set_title("자치구별 인구가중 접근성 (낮은 구부터)")
    axes[1].boxplot([result.loc[result.gu == g, "access"] for g in order.gu], positions=ypos,
                    orientation="horizontal", widths=.6, flierprops={"markersize": 3})
    axes[1].set_yticks(ypos, order.gu)
    axes[1].axvline(result.access.median(), color="#777777", linestyle="--", linewidth=.8)
    axes[1].set_title("구 안 행정동의 접근성 분포 (점선: 서울 중앙값)")
    for ax in axes:
        ax.set_xlabel("최근접역 중력형 접근성 (0~1)"); ax.spines[["top", "right"]].set_visible(False)
    fig.tight_layout(); fig.savefig(OUT / "gu_access.png", dpi=180); plt.close(fig)
    fig, axes = plt.subplots(1, 2, figsize=(15, 7))
    lim = float(max(result.mean_distance_line_m.max(), result.mean_distance_m.max())) * 1.05
    axes[0].scatter(result.mean_distance_line_m, result.mean_distance_m, s=14, alpha=.6, color="#24858d")
    axes[0].plot([0, lim], [0, lim], color="#777777", linestyle="--", linewidth=.8)
    axes[0].set(xlim=(0, lim), ylim=(0, lim), xlabel="인구가중 평균 직선거리 (m)", ylabel="인구가중 평균 보행망거리 (m)",
                title="동별 직선거리와 보행망거리 (점선: 같은 값)")
    result.plot(column="beyond_1km_pct", cmap="OrRd", ax=axes[1], legend=True, edgecolor="white", linewidth=.15,
                legend_kwds={"shrink": .7, "label": "%"})
    result.loc[result.beyond_1km_pct >= FAR_PCT].boundary.plot(ax=axes[1], color="#18342e", linewidth=1)
    axes[1].set_title(f"보행망 1km 밖 인구 비율 (검은 경계: {FAR_PCT}% 이상)"); axes[1].set_axis_off()
    fig.tight_layout(); fig.savefig(OUT / "walk_vs_line.png", dpi=180); plt.close(fig)

figures(result, stations, gu_table)
display(Image(filename=str(OUT / "overview.png")))
display(Image(filename=str(OUT / "scatter.png")))
display(Image(filename=str(OUT / "gu_access.png")))
display(Image(filename=str(OUT / "walk_vs_line.png")))

## 9. 결과 해석과 보고서

실제 계산값으로 보고서를 만듭니다. 관측 결과와 추정 원인을 구분하고 자료의 한계를 명시합니다.

In [ ]:
def report(result, candidates, s):
    a, cw, cl, nv = s["audit"], s["citywide"], s["citywide_line"], s["network_vs_line"]
    ca, ov, lu, lv = s["candidate_absolute"], s["overlap"], s["landuse"], s["living"]
    sens = {x["scenario"]: x for x in s["sensitivity"]}
    n_scn = s["scenario_count"]
    gu = pd.DataFrame(s["gu"])
    far, top = s["far_dongs"], s["top_beyond"]
    cand_names = set(candidates.name)
    yn = lambda flag: "포함" if flag else "미포함"
    pct_far_top = sum(x["beyond_1km_pop"] for x in top[:10]) / cw["beyond_1km_people"] * 100
    far_rows = "\n".join(
        f"|{x['name']}|{x['beyond_1km_pop']:,.0f}|{x['beyond_1km_pct']:.1f}|{x['density']:,.0f}|{x['access']:.3f}|"
        for x in top)
    cand_rows = "\n".join(
        f"|{x['name']}|{x['density']:,.0f}|{x['access']:.3f}|{x['mean_distance_m']:,.0f}|{x['beyond_1km_pct']:.0f}|{x['scenario_hits']}|"
        for x in candidates.to_dict("records"))
    scenario_rows = "\n".join(
        f"|{x['scenario']}|{x['candidate_count']}|{x['shared_with_baseline']}|{x['spearman_density_access']:.3f}|"
        for x in s["sensitivity"])
    top3 = result.nlargest(3, "access").name.tolist()
    bottom3 = result.nsmallest(3, "access").name.tolist()
    density_top = result.nlargest(3, "density").name.tolist()
    gu_far = gu.sort_values("beyond_1km_pct", ascending=False).head(3)
    gu_far_text = ", ".join(f"{x['gu']} {x['beyond_1km_pct']:.1f}%" for x in gu_far.to_dict("records"))
    gu_low = gu.sort_values("access").head(3)
    gu_high = gu.sort_values("access", ascending=False).head(3)
    gu_text = lambda df: ", ".join(f"{x['gu']}({x['access']:.3f})" for x in df.to_dict("records"))
    cand_gu = candidates.gu.value_counts()
    cand_gu_text = ", ".join(f"{g} {k}개" for g, k in cand_gu.head(2).items())
    strong = s["strong_candidates"]
    rest = candidates.loc[candidates.beyond_1km_pct < 50]
    case_text = "### 확대 사례\n"
    for c in case_maps:
        m = c["metrics"]
        case_text += (f"- {c['name']}: 보행망 평균 {m['mean_distance_m']:,.0f}m(직선 {m['mean_distance_line_m']:,.0f}m), "
            f"500m 이내 인구 {m['within_500m_pct']:.1f}%, 1km 밖 인구 {m['beyond_1km_pct']:.1f}%. "
            f"전체 면적 밀도 {m['density']:,.0f}명/km², 개발지역 면적 밀도 {m['density_built']:,.0f}명/km². "
            f"기존 후보 {yn(c['priority'])}, 개발지역 면적 밀도를 쓰면 후보 {yn(c['built_priority'])}.\n")
        rt = c["route"]
        pt = rt["parts"]
        case_text += (f"  - 예시 격자 {rt['grid_code']}(공개 인구 {rt['population']:,.0f}명)의 보행 경로: 격자→노드 {pt['grid_snap_m']:,.0f}m + 보행망 {pt['network_m']:,.0f}m "
            f"+ 노드→역 {pt['station_snap_m']:,.0f}m = {pt['walk_m']:,.0f}m. 직선거리는 {pt['line_m']:,.0f}m, 도착 역은 {rt['station_name']}({rt['station_line']})이다.\n")
    case_text += ("\n사례는 규칙으로 골랐다. 첫째는 후보 중 1km 밖 인구 비율이 가장 낮은 동, 둘째는 가장 높은 동, "
        "셋째는 전체 면적 밀도로는 후보가 아니지만 개발지역 면적 밀도로는 후보가 되는 동 가운데 개발지역 비율이 가장 낮은 동이다. "
        "`outputs/case_maps.png`와 앱의 확대 지도에 인구 격자·역·용도지역을 함께 그렸고, 점선 원은 기준 역에서 직선 500m·1km 범위이고, 기준 역은 직선 최근접역으로 공개 인구가 가장 많이 배정된 역이라 보행망 최근접역과 다를 수 있다. 주황 선은 예시 격자의 보행망 최단 경로다. "
        "지도만으로 경사·장벽·역 신설의 필요성을 확정하지 않는다.\n\n")
    text = f"""# 사람이 많이 사는 곳은 지하철도 가까울까?

## 한눈에 보기
공개 격자 인구의 {cw['within_1000m_pct']:.1f}%는 최근접역까지 OSM 보행망 거리로 1km 안에, {cw['within_500m_pct']:.1f}%는 500m 안에 산다. 1km 밖에 사는 사람은 약 {cw['beyond_1km_people'] / 1e4:,.0f}만 명({cw['beyond_1km_pct']:.1f}%)이다. 동 인구의 절반 이상이 1km 밖인 동은 {len(far)}개이고, 1km 밖 인구가 가장 많은 동은 {top[0]['name']}({top[0]['beyond_1km_pop']:,.0f}명)이다. 동별 인구밀도와 접근성의 순위상관은 ρ={s['spearman_rho']:.3f}로 약하다. 상대 순위로 뽑은 고밀도·저접근성 후보 {s['candidate_count']}개는 두 조건이 우연히 겹칠 때와 구별되지 않는다.

## 연구 질문과 대상
서울시 행정동별 인구밀도와 지하철 접근성은 어떤 관계이며, 고밀도·저접근성 지역은 어디인가? 2024년 공개 인구 격자를 2025년 6월 30일 기준 행정동 426개에 배정한다. 기준시점이 다른 수업 자료를 조합한 탐색 분석이며, 2026년 현재 교통 서비스를 평가하지 않는다.

## 자료와 전처리
- 인구: 수업 제공 `2024년_인구_다사_100M.csv`, 총인구 코드 `to_in_001`만 선택. 격자: `seoul_100m_grid_2025.geojson`. 원출처 표기는 수업 노트북에서 확인하지 못했고, 원본 수집일과 인구 비공개 처리 기준도 확인 못함.
- 경계: 수업 제공 `bnd_dong_seoul.shp`, 속성 BASE_DATE=20250630. SGIS 계열 행정동 코드. 법정동과 혼용하지 않는다.
- 역사: 수업 제공 서울시 역사마스터 CSV. [서울 열린데이터광장](https://data.seoul.go.kr/dataList/OA-21232/S/1/datasetView.do). 원본 수집일·서비스 기준일은 확인 못함. GTX {a['gtx_rows_excluded']}행 전체 제외. 개별 역의 영업 상태는 전수 확인하지 못했다.
- 보행망: OpenStreetMap 보행망(osmnx `network_type="walk"`)을 서울 경계 2.5km 버퍼 안에서 {a['walk_graph_downloaded_kst'][:10]}에 한 번 내려받아 `.cache/`에 저장하고 이후 캐시를 읽었다. 노드 {a['walk_graph_nodes']:,}개, 간선 {a['walk_graph_edges']:,}개. 격자 중심점과 역 좌표를 가장 가까운 노드에 연결하고 그 연결 거리를 더해 최단거리를 구했다. 4km를 넘는 값은 4km로 둔다. 연결 거리 때문에 직선거리보다 50m 넘게 짧아진 격자가 {a['walk_shorter_than_line_by_50m_grids']:,}개 있다.
- 용도지역: 수업 제공 `569_UQ111_용도지역(도시지역)_202609`의 `UPIS_C_UQ111.shp`. 다각형 {a['zoning_polygons']:,}개 중 대분류가 주거·상업·공업(UQA100·UQA200·UQA300)인 {a['zoning_built_polygons']:,}개를 개발지역으로 쓴다. 같은 부류의 중복 다각형이 있어 면적은 합집합으로 계산했고, 서울 안 개발지역은 {a['built_km2_total']:,.0f}km²이다. 파일명상 2026-09 기준이라 2025 행정동 경계와 시점이 다르다.
- 생활인구: 수업 제공 `lecture06_Living_Population/data/SPOP_2025_ADM_mid.csv`의 14시·03시 값. 행정동은 자치구명과 동명으로 연결했고(가운뎃점은 마침표로 통일) {a['living_pop_matched_dongs']}개 동이 일치했다. `SPOP_mid`는 수업 6주차 `02_living_population_time_series_visualization`에서 2025년 일별 시간대별 생활인구(서울 열린데이터광장 OA-23016, 수업 `01_living_population_data_preparation`이 월별 zip을 내려받아 합침)를 행정동·시간대별로 산술평균한 값이다. 요일·공휴일을 구분하지 않았고, 수업 노트북은 원자료에 결측이 많다고 적어 두었다.
- 분석일: {s['analysis_time_kst']}. 원본 파일 SHA-256과 보행망 캐시 해시는 `outputs/source_manifest.json`에 기록했다.
- 거리·면적은 EPSG:5179에서 계산. 원본 격자 {a['grid_before_population_join']:,}행을 고유 격자 키로 1:1 조인했고 이후 {a['grid_after_population_join']:,}행이다. 공개 인구값이 없는 격자는 {a['grids_with_missing_population']:,}개다. 해당 격자는 인구 가중치에 기여하지 않으며 실제 무인구로 단정하지 않는다.
- 격자 중심점을 행정동 내부에 배정했다. 공간조인 {a['grid_before_spatial_join']:,}행 → {a['grid_after_spatial_join']:,}행, 중복 0개. 중심점 미배정 {a['unassigned_grids']:,}개, 해당 공개 인구 합 {a['unassigned_population']:,.0f}명은 제외했다. 경계 격자 인구를 면적 비례로 나누지는 않았다.
- 서울 경계 2km 버퍼 안의 역·호선 좌표 점 {a['station_unique_coordinates_in_2km_buffer']}개(서울 안 {a['station_points_inside_seoul']}개)를 사용했다. 원본은 호선마다 좌표를 따로 가지므로 이 숫자는 역 수가 아니다. 완전히 같은 좌표만 중복 제거했고, 최근접역만 쓰는 기준 지표에서는 환승역 중복 가산이 없다.
- 행정동별 합산 뒤 경계와 1:1 조인: {s['dong_count']}행 유지. 공개 격자 인구 합 {s['public_population']:,}명을 보존했다. 이 합계를 서울시 공식 주민등록인구로 해석하지 않는다.

## 지표 정의
1. 인구밀도 D_d = Σ(P_i) / S_d. P_i는 동 d에 중심점이 속한 격자의 공개 인구, S_d는 행정동 전체 면적(km²)이다. 단위는 명/km². 값이 높으면 행정구역 전체 면적 대비 공개 격자 인구가 많다. 산·하천을 제외한 주거지 밀도가 아니다.
2. 최근접역 중력형 접근성 A_d = Σ[P_i × exp(−βt_i) × I(r_i≤2,000m)] / ΣP_i. r_i는 격자 i에서 최근접역까지 보행망 최단거리(m), t_i = r_i / (4,000/60 m/분), β=ln(2)/10. 역별 서비스 가중치는 1이다. 이는 전체 역의 기여도를 더한 접근성이 아니라 수업의 최근접역 변형이다. 점수 범위 0~1이며 높을수록 주민 분포가 가까운 역에 집중한다. 거리 0m에서 1, 약 667m에서 0.5, 2km에서 0.125, 2km 초과는 0이다. t_i는 거리 환산값이며 실제 보행시간이 아니다.
3. 절대 수준 지표: 인구가중 평균 보행망 거리(m), 500m 이내 인구 비율(%), 10분(667m) 이내 인구 비율(%), 1km 밖 인구 수와 비율. 직선거리로 계산한 같은 값을 비교용으로 함께 저장했다.
4. 역에서 먼 동: 공개 인구의 {s['far_threshold_pct']}% 이상이 1km 밖인 동. 50%는 이 분석에서 정한 기준이며 공식 기준이 아니다.
5. 밀도 분모 점검용 지표: 주거·상업·공업지역 면적 기준 밀도(명/km²), 주민가중 격자 밀도 Σ(P_i²)/ΣP_i(명/ha, 격자 한 칸이 1ha), 14시 생활인구 밀도(명/km²).

## 결과

### 서울 전체의 절대 수준
|기준|500m 이내(%)|1km 이내(%)|1km 밖 인구(명)|인구가중 평균 거리(m)|
|---|---:|---:|---:|---:|
|보행망|{cw['within_500m_pct']:.1f}|{cw['within_1000m_pct']:.1f}|{cw['beyond_1km_people']:,.0f}|{cw['weighted_mean_distance_m']:,.0f}|
|직선거리|{cl['within_500m_pct']:.1f}|{cl['within_1000m_pct']:.1f}|{cl['beyond_1km_people']:,.0f}|{cl['weighted_mean_distance_m']:,.0f}|

보행망 기준 10분(667m) 안에 역이 있는 주민은 {cw['within_10min_pct']:.1f}%다. 최근접역이 보행망 4km 안에 없는 주민은 {cw['unreachable_4km_pct']:.2f}%다. 평균 거리 계산에는 4km 상한이 적용되며, 상한이 적용된 공개 인구는 {cw['capped_4km_people']:,.0f}명이다. 아래에서 말하는 저접근성은 이 분포 안에서의 상대 순위다.

### 역에서 먼 동
1km 밖 인구가 많은 상위 10개 동에 1km 밖 인구의 {pct_far_top:.1f}%가 모여 있다. 인구의 절반 이상이 1km 밖인 동은 {len(far)}개다. 자치구별 1km 밖 인구 비율은 {gu_far_text} 순으로 높다. 상위 15개 동 중 고밀도·저접근성 후보에 든 동은 {sum(1 for x in top if x['name'] in cand_names)}개다.

|행정동|1km 밖 인구(명)|1km 밖 비율(%)|밀도(명/km²)|접근성|
|---|---:|---:|---:|---:|
{far_rows}

### 보행망과 직선거리
보행망 평균 거리는 직선거리의 {nv['mean_distance_ratio']:.2f}배다. 동별 접근성의 순위상관은 {nv['spearman_dong_access']:.3f}, 평균 거리의 순위상관은 {nv['spearman_dong_mean_distance']:.3f}이다. 인구의 절반 이상이 1km 밖인 동은 보행망으로 {nv['far_dongs_walk']}개, 직선거리로 {nv['far_dongs_line']}개다. 1km 밖 인구는 직선거리 기준 {cl['beyond_1km_people']:,.0f}명, 보행망 기준 {cw['beyond_1km_people']:,.0f}명이다. 격자와 역을 보행망 노드에 잇는 연결 거리(격자 중앙값 {a['grid_snap_median_m']:.0f}m, 역 중앙값 {a['station_snap_median_m']:.0f}m)를 빼고 노드 사이 거리만 재면 {nv['nosnap_beyond_1km_people']:,.0f}명({nv['nosnap_beyond_1km_pct']:.1f}%)이다. 보행망 기준 수치는 이 연결 거리 가정에 민감하다.

### 밀도와 접근성의 관계
426개 동에서 Pearson r={s['pearson_r']:.3f}(p={s['pearson_p']:.3f}), Spearman ρ={s['spearman_rho']:.3f}(p={s['spearman_p']:.3f})이다. 방향은 양이지만 ρ²가 {s['spearman_rho'] ** 2:.3f}여서 밀도 순위가 접근성 순위의 변동을 거의 설명하지 못한다. 이는 행정동 간 기술적 연관이며 인구 증가가 역 접근성을 높인다는 인과 증거가 아니다. 인구밀도 중앙값은 {s['median_density']:,.0f}명/km², 접근성 중앙값은 {s['median_access']:.3f}이다.

밀도의 분모를 바꿔도 연관은 약하다. 전체 면적 밀도는 개발지역 비율과 강하게 연결되어(ρ={lu['spearman_built_share_density']:.2f}) 산지와 하천이 큰 동에서 낮게 나온다. 개발지역 면적으로 나눈 밀도와 접근성의 ρ는 {sens['주거·상업·공업지역 면적 밀도']['spearman_density_access']:.3f}, 주민가중 격자 밀도와는 {sens['주민가중 격자 밀도']['spearman_density_access']:.3f}, 14시 생활인구 밀도와는 {sens['생활인구(14시) 밀도']['spearman_density_access']:.3f}이다.

### 후보 선별
인구밀도 상위 25%(≥{s['density_q75']:,.0f}명/km²)와 접근성 하위 25%(≤{s['access_q25']:.3f})를 동시에 만족하는 동은 {s['candidate_count']}개다. 두 집단이 서로 독립이면 겹치는 동은 평균 {ov['expected_if_independent']:.1f}개다. 접근성 값을 동 사이에서 섞는 순열 검정 10,000회에서 관측값({ov['observed']}개) 이상이 나온 비율은 {ov['permutation_p_at_least_observed']:.2f}다. 겹침이 우연으로 생기는 수와 구별되지 않으므로, 이 목록을 밀도가 높을수록 접근성이 낮다는 증거로 쓸 수 없다. 공식 교통 취약지역이나 사업 우선순위도 아니다.

|후보 지역|밀도(명/km²)|접근성|인구가중 보행망 거리(m)|1km 밖 인구(%)|후보로 선별된 조건 수(전체 {n_scn})|
|---|---:|---:|---:|---:|---:|
{cand_rows}

후보 동의 인구가중 보행망 거리는 중앙값 {ca['median_mean_distance_m']:,.0f}m, 최대 {ca['max_mean_distance_m']:,.0f}m다. 500m 이내 인구 비율의 중앙값은 {ca['median_within_500m_pct']:.1f}%, 1km 밖 인구 비율의 중앙값은 {ca['median_beyond_1km_pct']:.1f}%다. 인구의 절반 이상이 1km 밖인 후보는 {len(strong)}개({', '.join(strong) if strong else '없음'})이고, 나머지 {len(rest)}개는 1km 밖 인구가 {rest.beyond_1km_pct.max():.0f}% 이하다.

### 지역 차이와 도시 구조
인구밀도 상위 동은 {', '.join(density_top)}이다. 접근성이 높은 동은 {', '.join(top3)}, 낮은 동은 {', '.join(bottom3)}이다. 자치구별 인구가중 접근성이 낮은 구는 {gu_text(gu_low)}이고, 높은 구는 {gu_text(gu_high)}이다. 후보는 {cand_gu_text}로 두 구에 가장 많다. 접근성 하위 25% 동 {ov['low_count']}개 중 {ov['low_access_with_density_below_median']}개는 밀도가 서울 중앙값 이하다. 이런 동이 저밀도·저접근성 쪽에 있어 양의 상관에 기여한다.

개발지역 비율은 밀도와 강하게(ρ={lu['spearman_built_share_density']:.2f}), 접근성과는 약하게(ρ={lu['spearman_built_share_access']:.2f}) 양의 관계를 보인다. 후보 동의 개발지역 비율 중앙값은 {lu['median_built_share_candidates'] * 100:.0f}%로 서울 전체 중앙값 {lu['median_built_share_all'] * 100:.0f}%보다 높다. 14시 생활인구 밀도와 접근성의 ρ는 {lv['spearman_lp14_density_access']:.2f}로, 거주 인구밀도와 접근성의 ρ {s['spearman_rho']:.2f}보다 크다. 낮 시간에 사람이 몰리는 동이 역과 더 가깝다는 뜻이다. 14시 생활인구가 거주 공개 인구보다 많은 동의 비율은 {lv['share_day_over_resident'] * 100:.0f}%다.

{case_text}## 민감도
조건 {n_scn}가지로 후보를 다시 뽑았다. 반감기와 분위 기준만 바꾼 조건은 기준 후보를 대부분 포함하거나 포개진다. 직선거리를 쓰면 후보가 {sens['직선거리 사용']['candidate_count']}개이고 기준과 {sens['직선거리 사용']['shared_with_baseline']}개가 같다. 모든 역을 합산하면 점 단위 {sens['모든 역 합산 (점 단위 가산)']['shared_with_baseline']}개, 역사명 보정 {sens['모든 역 합산 (역사명 보정)']['shared_with_baseline']}개가 같다. 밀도 분모를 바꾸면 개발지역 면적 밀도에서 {sens['주거·상업·공업지역 면적 밀도']['shared_with_baseline']}/{sens['주거·상업·공업지역 면적 밀도']['candidate_count']}개, 주민가중 격자 밀도에서 {sens['주민가중 격자 밀도']['shared_with_baseline']}/{sens['주민가중 격자 밀도']['candidate_count']}개, 생활인구 밀도에서 {sens['생활인구(14시) 밀도']['shared_with_baseline']}/{sens['생활인구(14시) 밀도']['candidate_count']}개가 공통이다. 모든 조건에서 후보로 남은 동은 {', '.join(s['robust_all']) if s['robust_all'] else '없음'}이고, 세 가지 밀도 분모에서 모두 남은 동은 {', '.join(s['robust_density']) if s['robust_density'] else '없음'}이다. 상대 순위 후보는 밀도 정의에 민감하므로 개별 동의 순위를 근거로 삼기 어렵다. 1km 밖 인구를 쓰는 절대 기준 목록은 밀도 분모의 영향은 받지 않지만, 보행망과 연결 거리 가정의 영향은 받는다.

|조건|후보 동 수|기준 후보와 공통|Spearman ρ|
|---|---:|---:|---:|
{scenario_rows}

## 이 분석으로 설명할 수 없는 점
- 역과 주거지가 어긋나는 이유. 노선 배치, 건물 층수와 용도, 지형, 버스 노선 자료를 쓰지 않았다.
- 실제 체감 접근성. 경사, 계단, 신호, 야간 보행 안전, 역 출입구 위치는 반영하지 않았다.
- 동 안의 차이. 행정동 평균은 동 안의 격자 차이를 가린다. 확대 사례가 이를 일부 보여 줄 뿐이다.
- 시간대 차이. 거주 공개 인구를 가중치로 썼고, 생활인구는 14시와 03시 값을 밀도 점검에만 썼다.

## 해석의 한계
- 보행망 거리는 OSM 편집 상태에 따라 누락된 길이 있을 수 있고, 연결 거리 보정은 근사다. 철도 이용 후 목적지까지의 이동, 운행 빈도, 배차, 운임은 제외한다.
- 버스가 좋은 지역을 교통 소외 지역으로 단정할 수 없다. 환승 편의와 역별 서비스 차이를 반영하지 않는 최근접역 지표다. 모든 역 합산 조건의 점 단위 가산은 호선별 좌표를 모두 세므로 환승역이 여러 번 가산되고, 역사명 보정은 같은 역사명의 점을 하나로 나눠 센다. 호선 수로 서비스 수준을 가중하지는 않았다.
- 2024 인구·2025 경계·기준일 미확인 역 자료·파일명상 2026-09 용도지역·2025 생활인구를 결합했다. 2026년 현황이나 특정 연도의 완전한 교통망을 재현하지 않는다.
- 인구 비공개·결측 처리, 중심점 배정, 경계 변경, 전체 면적 분모로 인한 오차가 있다. 공식 동별 인구와의 외부 대조는 수행하지 않았다.
- 순열 검정은 이웃한 동의 값이 비슷한 공간 자기상관을 반영하지 않는다. p값은 참고로만 읽는다.
- 동별 상관관계에서 개인의 교통 경험을 추론하지 않는다. 후보 선별은 상대적 분위 기준이며 정책적 적정 기준이 아니다.

## 과제 조건 점검
- 연구 질문과 분석 대상: 위 연구 질문과 자료 절.
- 지표 2개 이상: 인구밀도, 보행망 최근접역 중력형 접근성. 보조로 1km 밖 인구, 10분 도달 인구, 개발지역 면적 밀도, 주민가중 격자 밀도, 생활인구 밀도.
- 자료와 전처리: 자료와 전처리 절, `outputs/source_manifest.json`.
- 공간 시각화와 비공간 시각화: 지도 `outputs/overview.png`, `outputs/walk_vs_line.png`, `outputs/case_maps.png`와 앱의 4종 지도. 산점도 `outputs/scatter.png`, 자치구 막대·상자그림 `outputs/gu_access.png`, 보행망·직선 비교 산점도.
- 주요 결과와 해석: 결과, 민감도, 이 분석으로 설명할 수 없는 점.
"""
    (ROOT / "REPORT.md").write_text(text)
    (DIST / "report.txt").write_text(text)


report(result, candidates, summary)
display(Markdown((ROOT / "REPORT.md").read_text()))


## 10. 로컬 웹 앱 생성

외부 라이브러리·API 키 없이 작동하는 정적 앱입니다. `dist/index.html`을 브라우저로 열면 지도·산점도·지역 선택·CSV 다운로드를 이용할 수 있습니다. 여기서는 배포하거나 제출하지 않습니다.

In [ ]:
HTML = r"""<!doctype html>
<html lang="ko"><head><meta charset="utf-8"><meta name="viewport" content="width=device-width,initial-scale=1"><title>사람이 많이 사는 곳은 지하철도 가까울까?</title>
<style>
:root{--paper:#f6f4ee;--ink:#18342e;--muted:#627169;--line:#d9dfd6;--teal:#1c7772;--red:#b34b37}*{box-sizing:border-box}body{margin:0;background:var(--paper);color:var(--ink);font-family:-apple-system,BlinkMacSystemFont,"Apple SD Gothic Neo",sans-serif;line-height:1.65}main{max-width:1400px;margin:auto;padding:32px 5vw 64px}header{display:flex;justify-content:space-between;font-size:13px;border-bottom:1px solid var(--ink);padding-bottom:15px}h1{font-size:clamp(32px,4.3vw,62px);line-height:1.18;letter-spacing:-.05em;margin:36px 0 20px}h2{font-size:25px;letter-spacing:-.03em;margin:0 0 8px}h3{margin:0;font-size:17px}p{margin:8px 0 16px}.intro{max-width:830px;color:var(--muted)}.eyebrow{letter-spacing:.12em;font-size:12px;margin-top:32px;color:var(--teal)}.stats{display:grid;grid-template-columns:repeat(4,1fr);gap:16px;margin:30px 0}.stat{border-top:2px solid var(--ink);padding:12px 0}.stat b{display:block;font-size:36px;font-weight:600;line-height:1.3}.stat span,.small{font-size:12px;color:var(--muted)}section{margin-top:40px}.toolbar{display:flex;flex-wrap:wrap;align-items:center;gap:10px;margin:20px 0}select,button,.link{font:inherit;border:1px solid var(--line);background:white;color:var(--ink);padding:8px 12px;border-radius:5px}select{max-width:100%}button{cursor:pointer}button:hover,a:hover{color:var(--teal)}.maps{display:grid;grid-template-columns:1fr 1fr;gap:20px}.card{background:#fff;border:1px solid var(--line);border-radius:8px;padding:20px;min-width:0}.card svg{width:100%;display:block;height:auto}.maparea path{stroke:#fff;stroke-width:.55;cursor:pointer}.maparea path:hover{stroke:var(--ink);stroke-width:2}.maparea path.selected{stroke:#b34b37;stroke-width:3}.maparea path.dim{opacity:.2}.legend{display:flex;gap:3px;align-items:center;font-size:11px;color:var(--muted);flex-wrap:wrap}.legend i{display:inline-block;width:28px;height:9px}.detail{margin-top:15px;border-left:4px solid var(--teal);padding:10px 18px;background:#e6eee7;min-height:72px}.detail b{font-size:18px}.analysis{display:grid;grid-template-columns:1.3fr 1fr;gap:20px}.analysis .card{padding:20px}.table-wrap{overflow:auto;max-height:440px}table{border-collapse:collapse;width:100%;font-size:12px}th,td{padding:10px 6px;border-bottom:1px solid var(--line);text-align:right;white-space:nowrap}th:first-child,td:first-child{text-align:left}tbody tr{cursor:pointer}tbody tr:hover{background:#eef4ee}.tag{color:var(--red)}.note{border-top:1px solid var(--line);padding-top:16px;color:var(--muted);font-size:13px}details{border-bottom:1px solid var(--line);padding:16px 0}summary{cursor:pointer;font-weight:600}a{color:var(--teal)}.downloads{display:flex;gap:10px;flex-wrap:wrap;margin-top:20px}.downloads a{font-size:13px;text-decoration:none}footer{font-size:12px;border-top:1px solid var(--ink);padding-top:16px;margin-top:40px;color:var(--muted)}.bars .row{display:grid;grid-template-columns:64px 1fr 230px;gap:8px;align-items:center;font-size:12px;margin:5px 0}.bars .track{background:#eef2ec;height:14px;border-radius:2px}.bars .fill{height:100%;background:var(--teal);border-radius:2px}.note table{margin-top:10px}.note th{white-space:normal}.src th,.src td{text-align:left;white-space:normal;vertical-align:top}.src code{font-size:11px}@media(max-width:800px){.bars .row{grid-template-columns:56px 1fr}.bars .row span:last-child{grid-column:1/-1}main{padding:20px}.maps,.analysis{grid-template-columns:1fr}.stats{grid-template-columns:1fr 1fr}.stat b{font-size:30px}header{font-size:11px}.card{padding:14px}}:focus-visible{outline:3px solid #c88631;outline-offset:2px}

.answer{border-left:4px solid var(--teal);padding:16px 22px;background:#e5eee6;max-width:940px;margin:24px 0}.answer strong{display:block;font-size:21px;line-height:1.5}.answer p{margin:8px 0 0}.type-legend{display:grid;gap:13px;font-size:13px}.type-legend span{display:flex;gap:9px;align-items:center}.type-legend i{width:15px;height:15px;flex-shrink:0}.joint-layout{display:grid;grid-template-columns:minmax(0,1fr) 260px;gap:25px;align-items:center}.joint-layout svg{max-height:560px}.maparea path.priority{stroke:#b34b37;stroke-width:1.2}.maparea path.selected{stroke:#18342e;stroke-width:3}.overview-stations circle{fill:#18342e;stroke:#fff;stroke-width:.5}.overview-stations{pointer-events:all}.case-buttons{display:grid;grid-template-columns:repeat(3,1fr);gap:12px;margin:22px 0}.case-button{text-align:left;background:#fff;padding:16px;border:1px solid var(--line);line-height:1.7}.case-button[aria-pressed=true]{border:2px solid var(--teal);background:#eaf1ea;padding:15px}.case-button strong{display:block;font-size:17px}.case-button span{display:block;font-size:12px;color:var(--muted)}.case-layout{display:grid;grid-template-columns:minmax(0,1.5fr) minmax(260px,1fr);gap:24px}.case-map{background:#f9faf7;touch-action:none;cursor:grab;max-height:650px}.case-map:active{cursor:grabbing}.case-controls{display:flex;gap:8px;flex-wrap:wrap;align-items:center;margin-bottom:10px}.case-controls button{padding:6px 12px}.case-metrics{display:grid;grid-template-columns:1fr 1fr;gap:12px;margin:18px 0}.case-metrics div{border-top:1px solid var(--line);padding-top:10px}.case-metrics b{font-size:24px;display:block}.case-metrics span{font-size:12px;color:var(--muted)}.case-legend{display:flex;flex-wrap:wrap;gap:8px;font-size:11px;margin:12px 0}.case-legend span{display:flex;align-items:center;gap:4px}.case-legend i{height:10px;width:14px;display:inline-block}.case-label{font-weight:600}.case-caption{font-size:12px;color:var(--muted)}.case-grid{cursor:pointer}.comparison-note{border-top:1px solid var(--line);padding-top:16px;font-size:13px}.toolbar label{font-size:13px}input[type=checkbox]{accent-color:var(--teal)}.mobile-scroll{overflow-x:auto}
@media(max-width:800px){.joint-layout,.case-layout{grid-template-columns:1fr}.case-buttons{grid-template-columns:1fr}.type-legend{grid-template-columns:1fr 1fr}.answer strong{font-size:18px}.case-map{min-height:300px}.case-button{padding:12px}.case-button[aria-pressed=true]{padding:11px}}
</style></head><body><main>
<header><span>스마트시티 융합기술세미나 · 과제 03</span><span>서울 / 426개 행정동 · <a href="https://github.com/ijin5/smartcity-assignment3-transit-access" target="_blank" rel="noopener">GitHub</a></span></header>
<div class="eyebrow">URBAN DENSITY × RAIL ACCESS</div>
<h1>사람이 많이 사는 곳은<br>지하철도 가까울까?</h1>
<p class="intro">인구가 많은 동일수록 지하철역도 가까울까요? 100m 격자에 사는 공개 인구를 가중치로, OSM 보행망 최단거리로 역까지의 거리를 재서 행정동별 인구밀도와 최근접역 접근성을 비교했습니다.</p>
<div class="answer"><strong id="answer-title"></strong><p id="answer-detail"></p><p class="small">상대 순위로 뽑은 후보와 별개로, 역에서 1km 넘게 떨어져 사는 인구를 절대 기준으로 따로 확인했습니다. 밀도의 분모를 바꾸면 상대 후보는 달라집니다.</p></div>
<p class="small">2024 공개 격자 인구 · 2025.06 행정동 경계 · 기준일 미확인 역사마스터 / OSM 보행망 최단거리(직선거리 비교) · GTX 제외</p>
<div class="stats"><div class="stat"><span>보행망 1km 밖 인구(공개 격자 인구)</span><b id="far-people"></b><span id="far-note">최근접역까지 보행망 1km 초과</span></div><div class="stat"><span>인구밀도 중앙값</span><b id="density"></b><span>명/km² · 행정동 전체 면적 기준</span></div><div class="stat"><span>순위상관계수 Spearman ρ</span><b id="rho"></b><span>두 지표의 연관은 약하게 나타남</span></div><div class="stat"><span>고밀도·저접근성 후보</span><b id="priority"></b><span id="priority-note">밀도 상위 25% ∩ 접근성 하위 25%</span></div></div>
<section><h2>01. 인구 분포와 역 접근성을 함께 보기</h2><p class="intro">지도는 각각의 지표를 5개 분위 구간으로 구분합니다. 색이 진할수록 값이 높습니다. 붉은 경계는 고밀도·저접근성 후보입니다. 동을 클릭하면 지도와 산점도에서 같은 지역을 확인할 수 있습니다.</p>
<div class="toolbar"><label for="gu">자치구</label><select id="gu"><option value="">서울 전체</option></select><label for="dong">행정동</label><select id="dong"><option value="">지역 선택</option></select><button id="reset">선택 초기화</button><label for="type-filter">결합 유형</label><select id="type-filter"><option value="">전체 유형</option><option value="0">고밀도·저접근성</option><option value="1">고밀도·그 외 접근성</option><option value="2">그 외 밀도·저접근성</option><option value="3">그 외 밀도·그 외 접근성</option></select><label><input id="show-stations" type="checkbox"> 역 위치 표시</label></div>
<div class="maps"><article class="card"><h3>인구밀도</h3><p class="small">공개 격자 인구 합 ÷ 행정동 전체 면적 · 명/km²</p><svg id="density-map" viewBox="0 0 620 510" role="img" aria-label="서울 행정동별 인구밀도 지도"></svg><div id="density-legend" class="legend"></div></article><article class="card"><h3>최근접역 중력형 접근성</h3><p class="small">주민 분포로 가중한 보행망 거리 감쇠 점수 · 0~1</p><svg id="access-map" viewBox="0 0 620 510" role="img" aria-label="서울 행정동별 최근접역 접근성 지도"></svg><div id="access-legend" class="legend"></div></article></div>
<article class="card" style="margin-top:20px"><h3>두 지표를 결합한 유형 지도</h3><p class="small" id="type-thresholds"></p><div class="joint-layout"><svg id="joint-map" viewBox="0 0 620 510" role="img" aria-label="서울 행정동별 인구밀도와 접근성 결합 유형 지도"></svg><div><div id="type-legend" class="type-legend"></div><p class="note">고밀도는 밀도 상위 25%, 저접근성은 접근성 하위 25%입니다. ‘그 외’에는 중간값도 포함됩니다. 유형은 공식 교통 취약 등급이 아닙니다.</p></div></div></article>
<div id="detail" class="detail" aria-live="polite">지도·산점도·목록에서 지역을 선택하세요.</div></section>
<section id="far-section"><h2>02. 역에서 먼 사람들</h2><p class="intro" id="far-intro"></p><div class="analysis"><article class="card"><h3>보행망 1km 밖 인구 비율</h3><p class="small">최근접역까지 보행망 거리가 1km를 넘는 공개 인구 ÷ 동 공개 인구 · %</p><svg id="far-map" viewBox="0 0 620 510" role="img" aria-label="서울 행정동별 보행망 1km 밖 인구 비율 지도"></svg><div id="far-legend" class="legend"></div></article><article class="card"><h3>1km 밖에 사는 인구가 많은 동</h3><p class="small">보행망 기준 · 1km 밖 인구 내림차순 15개 동</p><div class="table-wrap"><table><thead><tr><th>행정동</th><th>1km 밖 인구</th><th>비율</th><th>접근성</th></tr></thead><tbody id="far-table"></tbody></table></div></article></div><p class="note" id="walk-vs-line"></p></section>
<section><h2>03. 밀도만으로 접근성을 설명할 수 있을까?</h2><p id="correlation" class="intro"></p><div class="analysis"><article class="card"><h3>행정동별 밀도와 접근성</h3><svg id="scatter" viewBox="0 0 720 490" role="img" aria-label="인구밀도와 접근성 산점도"></svg><p class="small">붉은 점: 고밀도·저접근성 후보 / 점선: 밀도 75백분위·접근성 25백분위</p></article><article class="card"><h3>추가 검토 후보</h3><p class="small">인구밀도 내림차순. 자치구 선택이 목록에도 적용됩니다.</p><div class="table-wrap"><table><thead><tr><th>행정동</th><th>밀도</th><th>접근성</th><th>거리(m)</th></tr></thead><tbody id="table"></tbody></table></div></article></div></section>
<section id="case-section"><h2>04. 동 평균 뒤의 세 가지 사례</h2><p class="intro">같은 후보 안에서도 역까지의 거리는 다릅니다. 인구 격자와 역의 배치를 확대해 보고, 행정동 면적에 따라 가려지는 사례도 비교합니다.</p><div id="case-buttons" class="case-buttons"></div><article class="card"><div class="case-layout"><div><div class="case-controls"><button id="case-zoom-in" aria-label="사례 지도 확대">＋</button><button id="case-zoom-out" aria-label="사례 지도 축소">−</button><button id="case-reset">사례 지도 전체 보기</button><label for="case-station">거리 원의 기준 역</label><select id="case-station"></select></div><svg id="case-map" class="case-map" viewBox="0 0 800 650" role="img" aria-label="선택 사례의 인구 격자와 지하철역 확대 지도"></svg><div class="case-legend"><span><i style="background:#f6dfa0"></i>1~99명</span><span><i style="background:#dbad55"></i>100~299명</span><span><i style="background:#b87932"></i>300~599명</span><span><i style="background:#784b24"></i>600명 이상</span><span><i style="background:repeating-linear-gradient(45deg,#ddd,#ddd 2px,#fafafa 2px,#fafafa 4px)"></i>인구 미수록</span><span><i style="background:#dcebe0"></i>주거·상업·공업지역</span></div><p class="case-caption">격자 한 칸은 100m × 100m입니다. 점은 역·호선 좌표입니다. 동 바깥의 역도 표시합니다. 격자를 누르면 직선·보행망 거리를 확인할 수 있습니다. 주황 선은 예시 격자의 보행망 최단 경로입니다. 드래그로 이동할 수 있습니다.</p><p id="case-grid-detail" class="case-caption" aria-live="polite"></p></div><div><p class="eyebrow" id="case-kicker"></p><h3 id="case-title"></h3><p id="case-story"></p><div id="case-metrics" class="case-metrics"></div><p id="case-density-note" class="comparison-note"></p><p id="case-ring-note" class="case-caption"></p><p id="case-route" class="case-caption"></p><p class="note">원은 직선 최근접역 배정으로 고른 기준 역에서 직선 500m·1km 범위입니다. 보행망 최근접역 판정이나 실제 보행권과 다릅니다. 미수록 인구를 실제 무인구로 단정하지 않습니다. 지도만으로 경사·장벽·인과관계를 확인할 수는 없습니다.</p></div></div></article></section>
<section><h2>05. 자치구별 비교</h2><p class="intro">인구가중 접근성이 낮은 구부터 정렬했습니다. 막대는 0~1 척도이고, 오른쪽 숫자는 접근성, 최근접역이 보행망 1km 밖인 인구 비율, 후보 동 수입니다.</p><div class="card bars" id="gu-bars"></div></section>
<section><h2>06. 결과가 말해주는 것</h2><p id="findings"></p><p>고밀도·저접근성 후보는 두 지표가 각각 상·하위인 동이 겹친 목록입니다. 겹침 수가 우연 수준이므로 밀도가 높을수록 접근성이 낮다는 증거로 쓸 수 없습니다. 노선망·건물 용도·지형·버스를 분석하지 않았으므로 교통 소외나 역 신설 필요성을 확정할 수는 없습니다.</p><div id="sensitivity" class="note"></div></section>
<section><h2>분석 방법과 읽을 때의 주의점</h2>
<details open><summary>두 지표의 계산식과 변수</summary><p>인구밀도 D<sub>d</sub> = ΣP<sub>i</sub> / S<sub>d</sub>. P<sub>i</sub>는 동에 배정된 격자의 공개 인구, S<sub>d</sub>는 행정동 전체 면적(km²)입니다. 산·하천을 제외한 주거지 밀도가 아닙니다.</p><p>접근성 A<sub>d</sub> = Σ[P<sub>i</sub> × exp(−βt<sub>i</sub>) × I(t<sub>i</sub> ≤ 30)] / ΣP<sub>i</sub>. t<sub>i</sub>는 최근접역까지 OSM 보행망 최단거리(m)를 4,000/60(m/분)으로 나눈 값, β = ln(2)/10입니다. 역별 가중치는 1이며 최근접역 한 곳만 반영합니다.</p><p>접근성은 0m에서 1, 약 667m에서 0.5, 2km에서 0.125, 2km를 넘으면 0입니다. 높은 점수는 주민 분포가 역에 가깝다는 뜻입니다. 시속 4km는 거리 환산을 위한 가정이며 실제 보행시간이 아닙니다.</p><p>보조 지표: 주거·상업·공업지역 합집합 면적으로 나눈 밀도, 주민가중 격자 밀도 ΣP<sub>i</sub>²/ΣP<sub>i</sub>(명/ha), 최근접역이 보행망 1km 밖인 인구 수와 비율, 직선거리 비교값, 14시 생활인구 밀도입니다. 전체 면적 밀도가 산지·하천의 영향을 받는지, 직선거리가 접근성을 낙관하는지 점검하려고 넣었습니다.</p></details>
<details><summary>데이터 출처·전처리·조인 검증</summary><p>수업 제공 인구·격자·행정동 경계·용도지역(파일명상 2026-09 기준)과 서울시 역사마스터 CSV, 2025 생활인구(수업 7주차 파일)를 재사용했고, OSM 보행망은 Overpass에서 한 번 내려받아 캐시했습니다. 인구 원출처와 세부 비공개 처리 기준은 추가 확인이 필요합니다. 자료 원본 수집일과 역 목록 기준일은 확인 못함입니다. <a href="https://github.com/jslee-gses/code" target="_blank" rel="noopener">수업 저장소</a> · <a href="https://data.seoul.go.kr/dataList/OA-21232/S/1/datasetView.do" target="_blank" rel="noopener">서울시 역사마스터</a> · <a href="https://sgis.kostat.go.kr/" target="_blank" rel="noopener">SGIS(행정동 코드 체계 참고)</a></p><p id="audit"></p><div class="table-wrap"><table class="src"><thead><tr><th>자료</th><th>제공처·출처</th><th>기준 시점</th><th>수집일·확인 상태</th></tr></thead><tbody><tr><td>100m 격자 인구 <code>2024년_인구_다사_100M.csv</code></td><td>수업 제공. 원출처 표기는 수업 노트북에서 확인하지 못함</td><td>2024</td><td>수집일 확인 못함</td></tr><tr><td>100m 격자 경계 <code>seoul_100m_grid_2025.geojson</code></td><td>수업 제공</td><td>2025</td><td>확인 못함</td></tr><tr><td>행정동 경계 <code>bnd_dong_seoul.shp</code></td><td>수업 제공, SGIS 계열 행정동 코드</td><td>BASE_DATE 2025-06-30</td><td>수집일 확인 못함</td></tr><tr><td>서울시 역사마스터</td><td>서울 열린데이터광장 OA-21232, 수업 제공 CSV</td><td>기준일 확인 못함</td><td>수집일 확인 못함</td></tr><tr><td>OSM 보행망</td><td>© OpenStreetMap contributors (ODbL), Overpass, osmnx 2.1.1</td><td>수집 시점의 편집 상태</td><td id="osm-date"></td></tr><tr><td>용도지역(도시지역)</td><td>수업 제공 <code>569_UQ111_용도지역(도시지역)_202609</code></td><td>파일명상 2026-09</td><td>확인 못함</td></tr><tr><td>생활인구 <code>SPOP_2025_ADM_mid.csv</code></td><td>서울 열린데이터광장 OA-23016를 수업 6주차 노트북이 가공</td><td>2025년 일별·시간대별 값의 산술평균(요일·공휴일 구분 없음)</td><td>원자료에 결측 많음(수업 노트북 지적)</td></tr></tbody></table></div><p>총인구 코드 to_in_001을 고유 격자 키로 1:1 조인했습니다. EPSG:5179에서 중심점으로 행정동을 배정하고 거리·면적을 계산했습니다. 정확히 같은 역 좌표를 중복 제거하고 GTX 전체를 제외했습니다. 경계 효과를 줄이기 위해 서울 밖 2km 버퍼의 역도 포함했습니다. 역별 영업 상태는 전수 확인하지 못했습니다. 격자 중심점과 역 좌표를 가장 가까운 보행망 노드에 연결하고 그 연결 거리를 더해 최단거리를 구했으며, 4km를 넘는 값은 4km로 두었습니다.</p></details>
<details open><summary>해석의 한계</summary><p>보행망 거리는 경사·신호·횡단 가능 여부·계단·출입구 위치를 반영하지 않으며, OSM에 누락된 길이 있을 수 있습니다. 버스, 배차, 운임, 목적지까지의 통행시간도 제외합니다. 전체 대중교통 접근성으로 해석하지 않습니다.</p><p>인구·경계·역 자료의 기준시점이 다릅니다. 2026년 현재 현황이나 특정 연도의 완전한 교통망 재현이 아닙니다. 미수록 인구는 실제 무인구로 단정하지 않습니다. 중심점 배정으로 동 경계의 인구가 일부 제외됩니다.</p><p>상관은 행정동 간 기술적 연관이며 인과관계가 아닙니다. 개인의 교통 경험을 추론하지 않습니다. 후보는 상대적 사분위 기준이며 공식 취약지역·사업 우선순위가 아닙니다.</p><p>용도지역은 파일명상 2026-09 기준이고 행정동 경계는 2025.06 기준이라 시점이 다릅니다. 순열 검정은 이웃한 동의 값이 비슷한 공간 자기상관을 반영하지 않아 p값은 참고용입니다. 버스·노선 운행 자료는 쓰지 않았습니다.</p></details>
<details><summary>이 분석으로 설명할 수 없는 점</summary><p>역과 주거지가 어긋나는 이유는 노선 배치, 건물 층수와 용도, 지형, 버스 노선 자료가 없어 설명하지 못합니다. 행정동 평균은 동 안의 격자 차이를 가립니다. 생활인구는 14시와 03시 값을 밀도 점검에만 썼습니다.</p></details>
<details><summary>과제 조건 점검</summary><p>연구 질문과 분석 대상: 첫 화면과 보고서 첫 절. 지표 2개 이상: 인구밀도, 보행망 최근접역 중력형 접근성, 보조로 1km 밖 인구·개발지역 면적 밀도·주민가중 격자 밀도·생활인구 밀도. 자료와 전처리: 위의 데이터 출처 접기와 원본 검증 기록. 공간 시각화: 지도 4종(밀도·접근성·결합 유형·1km 밖 비율)과 사례 확대 지도. 비공간 시각화: 산점도, 자치구 막대, 민감도 표. 주요 결과와 해석: 06절과 보고서.</p></details>
<div class="downloads"><a class="link" href="dong_indicators.csv" download>전체 동별 지표 CSV</a><a class="link" href="priority_areas.csv" download>후보 지역 CSV</a><a class="link" href="gu_summary.csv" download>자치구 요약 CSV</a><a class="link" href="report.txt" download>분석 보고서</a><a class="link" href="source_manifest.json" download>원본 파일 검증 기록</a><a class="link" href="https://github.com/ijin5/smartcity-assignment3-transit-access" target="_blank" rel="noopener">GitHub 저장소</a><a class="link" href="https://github.com/ijin5/smartcity-assignment3-transit-access/blob/main/assignment3_transit_access.ipynb" target="_blank" rel="noopener">분석 노트북</a></div></section>
<footer id="footer"></footer></main><script src="data.js"></script><script src="cases.js"></script><script>
'use strict';
const {summary:s,geo}=window.STUDY, features=geo.features, ns='http://www.w3.org/2000/svg', n=x=>Math.round(x).toLocaleString('ko-KR'), $=id=>document.getElementById(id);
const props=features.map(f=>f.properties), byId=new Map(props.map(p=>[p.ADM_CD,p]));let selected='';
function el(tag,attrs={},text=''){const node=document.createElementNS(ns,tag);Object.entries(attrs).forEach(([k,v])=>node.setAttribute(k,v));node.textContent=text;return node;}
$('density').textContent=n(s.median_density);$('rho').textContent=s.spearman_rho.toFixed(3);$('priority').textContent=s.candidate_count+'개';
$('answer-title').textContent=`공개 격자 인구의 ${s.citywide.within_1000m_pct.toFixed(0)}%는 최근접역까지 보행망 1km 안에 삽니다. 1km 밖에는 약 ${Math.round(s.citywide.beyond_1km_people/10000)}만 명이 살고, 동 인구의 절반 이상이 1km 밖인 동은 ${s.far_dongs.length}개입니다.`;$('answer-detail').textContent=`동별 인구밀도와 접근성의 순위상관은 ${s.spearman_rho.toFixed(3)}로 약합니다. 고밀도·저접근성 후보 ${s.candidate_count}개는 두 기준의 교집합이며, 우연한 겹침보다 많다는 근거는 없습니다.`;$('far-people').textContent=Math.round(s.citywide.beyond_1km_people/10000)+'만 명';$('far-note').textContent=`서울 공개 인구의 ${s.citywide.beyond_1km_pct.toFixed(1)}% · 직선거리로 재면 ${Math.round(s.citywide_line.beyond_1km_people/10000)}만 명`;$('far-intro').textContent=`공개 격자 인구 기준 보행망 ${n(s.citywide.beyond_1km_people)}명이 최근접역에서 1km 밖에 삽니다. 동 인구의 절반 이상이 1km 밖인 동은 ${s.far_dongs.length}개입니다. 아래 표는 1km 밖 인구가 많은 동입니다.`;$('walk-vs-line').textContent=`보행망 평균 거리는 직선거리의 ${s.network_vs_line.mean_distance_ratio.toFixed(2)}배입니다. 동별 접근성 순위상관은 ${s.network_vs_line.spearman_dong_access.toFixed(3)}이고, 절반 이상이 1km 밖인 동은 보행망 ${s.network_vs_line.far_dongs_walk}개, 직선거리 ${s.network_vs_line.far_dongs_line}개입니다. 노드 연결 거리를 빼고 재면 1km 밖 인구는 ${Math.round(s.network_vs_line.nosnap_beyond_1km_people/10000)}만 명입니다. 평균 거리에는 4km 상한이 적용되며 상한 적용 인구는 ${n(s.citywide.capped_4km_people)}명입니다.`;
$('correlation').textContent=`Pearson r=${s.pearson_r.toFixed(3)} (p=${s.pearson_p.toFixed(3)}), Spearman ρ=${s.spearman_rho.toFixed(3)} (p=${s.spearman_p.toFixed(3)}). 동별 인구밀도와 접근성의 연관은 약하게 나타났습니다. 높은 인구밀도만으로 역 접근성이 높다고 보기 어렵습니다.`;
const ranked=[...props].sort((a,b)=>b.access-a.access);$('findings').textContent=`서울 주민의 ${s.citywide.within_500m_pct.toFixed(0)}%가 최근접역 보행망 500m 이내에, ${s.citywide.within_1000m_pct.toFixed(0)}%가 1km 이내에 삽니다. 접근성이 높은 동은 ${ranked.slice(0,3).map(p=>p.name).join(', ')}, 낮은 동은 ${ranked.slice(-3).reverse().map(p=>p.name).join(', ')}입니다. 밀도 ${n(s.density_q75)}명/km² 이상과 접근성 ${s.access_q25.toFixed(3)} 이하를 함께 만족한 동은 ${s.candidate_count}개입니다. 두 조건이 독립일 때 겹칠 것으로 기대되는 수는 ${s.overlap.expected_if_independent.toFixed(1)}개이고, 순열검정에서 관측 이상이 나올 확률은 p=${s.overlap.permutation_p_at_least_observed.toFixed(2)}입니다. 후보 중 인구의 절반 이상이 1km 밖에 사는 동은 ${s.strong_candidates.length}개(${s.strong_candidates.join(', ')})입니다.`;
$('priority-note').textContent=`밀도 상위 25% ∩ 접근성 하위 25% · 독립일 때 기대 ${s.overlap.expected_if_independent.toFixed(1)}개`;
(()=>{const box=$('sensitivity'),cap=document.createElement('p'),table=document.createElement('table'),head=document.createElement('tr');cap.textContent='민감도: 조건을 바꿨을 때 후보가 얼마나 유지되는지 확인했습니다. 모든 조건에서 후보로 남은 동: '+(s.robust_all.join(', ')||'없음')+'.';['조건','후보 수','기준과 공통','Spearman ρ'].forEach(t=>{const th=document.createElement('th');th.textContent=t;head.append(th);});table.append(head);s.sensitivity.forEach(x=>{const tr=document.createElement('tr');[x.scenario,x.candidate_count,x.shared_with_baseline,x.spearman_density_access.toFixed(3)].forEach(t=>{const td=document.createElement('td');td.textContent=t;tr.append(td);});table.append(tr);});box.append(cap,table);})();
s.gu.forEach(g=>{const row=document.createElement('div'),name=document.createElement('span'),track=document.createElement('div'),fill=document.createElement('div'),val=document.createElement('span');row.className='row';name.textContent=g.gu;track.className='track';fill.className='fill';fill.style.width=(g.access*100).toFixed(1)+'%';track.append(fill);val.textContent=`${g.access.toFixed(3)} · 1km 밖 ${g.beyond_1km_pct.toFixed(0)}% · 후보 ${g.candidates}`;row.append(name,track,val);$('gu-bars').append(row);});
const a=s.audit;$('audit').textContent=`격자 ${n(a.grid_before_population_join)}행 → 인구 조인 ${n(a.grid_after_population_join)}행 → 공간 조인 ${n(a.grid_after_spatial_join)}행. 공개 인구 미수록 ${n(a.grids_with_missing_population)}격자. 중심점 미배정 ${n(a.unassigned_grids)}격자·공개 인구 ${n(a.unassigned_population)}명 제외. 최종 ${n(s.public_population)}명 집계. 사용 역·호선 좌표 점 ${a.station_unique_coordinates_in_2km_buffer}개(서울 안 ${a.station_points_inside_seoul}개). 보행망 노드 ${n(a.walk_graph_nodes)}개·간선 ${n(a.walk_graph_edges)}개(OSM ${a.walk_graph_downloaded_kst.slice(0,10)} 수집). 생활인구 결합 ${a.living_pop_matched_dongs}개 동.`;
$('footer').textContent=`분석 실행 ${s.analysis_time_kst} · 로컬 수업 자료 재사용 · 보행망 © OpenStreetMap contributors · 스마트시티 융합기술세미나 과제 3`;$('osm-date').textContent=s.audit.walk_graph_downloaded_kst.slice(0,10)+' 수집';
const guSelect=$('gu'),dongSelect=$('dong'),typeSelect=$('type-filter');[...new Set(props.map(p=>p.gu))].sort().forEach(g=>guSelect.add(new Option(g,g)));
function updateOptions(){dongSelect.length=1;props.filter(p=>!guSelect.value||p.gu===guSelect.value).sort((a,b)=>a.name.localeCompare(b.name,'ko')).forEach(p=>dongSelect.add(new Option(p.name,p.ADM_CD)));}
const xy=[];features.forEach(f=>{const polygons=f.geometry.type==='Polygon'?[f.geometry.coordinates]:f.geometry.coordinates;polygons.forEach(poly=>poly.forEach(ring=>xy.push(...ring)));});
const minX=Math.min(...xy.map(v=>v[0])),maxX=Math.max(...xy.map(v=>v[0])),minY=Math.min(...xy.map(v=>v[1])),maxY=Math.max(...xy.map(v=>v[1]));
const scale=Math.min(580/(maxX-minX),470/(maxY-minY)),ox=(620-(maxX-minX)*scale)/2,oy=(510-(maxY-minY)*scale)/2;
function path(f){const polys=f.geometry.type==='Polygon'?[f.geometry.coordinates]:f.geometry.coordinates;return polys.map(poly=>poly.map(r=>r.map((v,i)=>(i?'L':'M')+(ox+(v[0]-minX)*scale).toFixed(1)+','+(oy+(maxY-v[1])*scale).toFixed(1)).join('')+'Z').join('')).join('');}
function quantile(values,q){const v=[...values].sort((a,b)=>a-b),i=(v.length-1)*q,lo=Math.floor(i);return v[lo]+(v[Math.ceil(i)]-v[lo])*(i-lo);}
function drawMap(id,key,colors){const svg=$(id),group=el('g',{'class':'maparea'});svg.append(group);const breaks=[.2,.4,.6,.8].map(q=>quantile(props.map(p=>p[key]),q));features.forEach(f=>{const p=f.properties,bin=breaks.filter(b=>p[key]>b).length,node=el('path',{d:path(f),fill:colors[bin],'fill-rule':'evenodd','data-id':p.ADM_CD,'class':p.priority?'priority':'',tabindex:0,'aria-label':p.name});node.append(el('title',{},`${p.name}: ${key==='density'?n(p[key])+'명/km²':p[key].toFixed(3)}`));node.addEventListener('click',()=>choose(p.ADM_CD));node.addEventListener('keydown',e=>{if(e.key==='Enter'){choose(p.ADM_CD);}});group.append(node);});const legend=$(key+'-legend');legend.append('낮음 ');colors.forEach(c=>{const i=document.createElement('i');i.style.background=c;legend.append(i);});legend.append(' 높음 · 경계값 '+breaks.map(b=>key==='density'?n(b):b.toFixed(2)).join(' / '));}
drawMap('density-map','density',['#f7edcf','#ead19a','#d7ad63','#b67d36','#855123']);drawMap('access-map','access',['#e0f0e8','#b1d7cd','#79b7af','#448e8b','#155e63']);
const svg=$('scatter'),left=70,plotTop=28,w=620,h=380,maxD=Math.ceil(Math.max(...props.map(p=>p.density))/10000)*10000,sx=d=>left+w*d/maxD,sy=d=>plotTop+h*(1-d);
for(let i=0;i<=5;i++){let x=i*maxD/5,y=i/5;svg.append(el('line',{x1:sx(x),x2:sx(x),y1:plotTop,y2:plotTop+h,stroke:'#edf0ea'}));svg.append(el('text',{x:sx(x),y:plotTop+h+22,'text-anchor':'middle','font-size':11,fill:'#627169'},n(x)));svg.append(el('text',{x:left-12,y:sy(y)+4,'text-anchor':'end','font-size':11,fill:'#627169'},y.toFixed(1)));}
svg.append(el('text',{x:left+w/2,y:465,'text-anchor':'middle','font-size':13},'인구밀도 (명/km²)'));svg.append(el('text',{x:70,y:16,'font-size':12},'접근성 (0~1)'));
svg.append(el('rect',{x:sx(s.density_q75),y:sy(s.access_q25),width:sx(maxD)-sx(s.density_q75),height:plotTop+h-sy(s.access_q25),fill:'#f9ece7'}));
svg.append(el('line',{x1:sx(s.density_q75),x2:sx(s.density_q75),y1:plotTop,y2:plotTop+h,stroke:'#8b938a','stroke-dasharray':'5 5'}));svg.append(el('line',{x1:left,x2:left+w,y1:sy(s.access_q25),y2:sy(s.access_q25),stroke:'#8b938a','stroke-dasharray':'5 5'}));
props.forEach(p=>{const node=el('circle',{cx:sx(p.density),cy:sy(p.access),r:4,fill:p.priority?'#b34b37':'#348983',opacity:.7,'data-id':p.ADM_CD,tabindex:0,'aria-label':p.name});node.append(el('title',{},`${p.name} / 밀도 ${n(p.density)} / 접근성 ${p.access.toFixed(3)}`));node.addEventListener('click',()=>choose(p.ADM_CD));node.addEventListener('keydown',e=>{if(e.key==='Enter')choose(p.ADM_CD);});svg.append(node);});
function updateTable(){const rows=props.filter(p=>p.priority&&(!guSelect.value||p.gu===guSelect.value)).sort((a,b)=>b.density-a.density);$('table').replaceChildren();rows.forEach(p=>{const tr=document.createElement('tr');[p.name,n(p.density),p.access.toFixed(3),n(p.mean_distance_m)].forEach(t=>{const td=document.createElement('td');td.textContent=t;tr.append(td);});tr.tabIndex=0;tr.addEventListener('click',()=>choose(p.ADM_CD));tr.addEventListener('keydown',e=>{if(e.key==='Enter')choose(p.ADM_CD);});$('table').append(tr);});if(!rows.length){const tr=document.createElement('tr'),td=document.createElement('td');td.colSpan=4;td.textContent='이 자치구에는 기준을 만족한 후보가 없습니다.';tr.append(td);$('table').append(tr);}}
function highlight(){document.querySelectorAll('[data-id]').forEach(node=>{const p=byId.get(node.dataset.id),dim=(guSelect.value&&p.gu!==guSelect.value)||(typeSelect.value!==''&&kind(p)!==Number(typeSelect.value));node.classList.toggle('dim',Boolean(dim));if(node.tagName==='path')node.classList.toggle('selected',p.ADM_CD===selected);if(node.tagName==='circle'){node.setAttribute('r',p.ADM_CD===selected?8:4);node.setAttribute('opacity',dim?.1:.75);node.setAttribute('stroke',p.ADM_CD===selected?'#18342e':'none');}});}
function choose(id){selected=id;const p=byId.get(id);if(!p)return;typeSelect.value='';guSelect.value=p.gu;updateOptions();dongSelect.value=id;updateTable();highlight();$('detail').replaceChildren();const title=document.createElement('b');title.textContent=p.name+(p.priority?' · 고밀도·저접근성 후보':'');$('detail').append(title,document.createElement('br'),`인구밀도 ${n(p.density)}명/km² / 접근성 ${p.access.toFixed(3)} / 인구가중 보행망 거리 ${n(p.mean_distance_m)}m(직선 ${n(p.mean_distance_line_m)}m) / 보행망 500m 이내 공개 인구 ${p.within_500m_pct.toFixed(1)}% / 보행망 1km 밖 ${p.beyond_1km_pct.toFixed(1)}%`);}
guSelect.addEventListener('change',()=>{selected='';updateOptions();updateTable();highlight();$('detail').textContent=guSelect.value?guSelect.value+'의 행정동을 선택하세요.':'지도·산점도·목록에서 지역을 선택하세요.';});dongSelect.addEventListener('change',()=>{if(dongSelect.value)choose(dongSelect.value);});$('reset').addEventListener('click',()=>{selected='';guSelect.value='';typeSelect.value='';updateOptions();updateTable();highlight();$('detail').textContent='지도·산점도·목록에서 지역을 선택하세요.';});updateOptions();updateTable();

// 두 지표의 교집합을 네 가지 배타적 유형으로 표시한다. '그 외'는 높음/낮음의 반대가 아니다.
const typeColors=['#b34b37','#d5ab59','#5b8790','#dce3db'];
const typeLabels=['고밀도·저접근성','고밀도·그 외 접근성','그 외 밀도·저접근성','그 외 밀도·그 외 접근성'];
function kind(p){return p.density>=s.density_q75?(p.access<=s.access_q25?0:1):(p.access<=s.access_q25?2:3);}
function drawJoint(){const group=el('g',{'class':'maparea'});$('joint-map').append(group);features.forEach(f=>{const p=f.properties,node=el('path',{d:path(f),fill:typeColors[kind(p)],'fill-rule':'evenodd','data-id':p.ADM_CD,'class':p.priority?'priority':'',tabindex:0,'aria-label':p.name});node.append(el('title',{},p.name+' / '+typeLabels[kind(p)]));node.addEventListener('click',()=>choose(p.ADM_CD));node.addEventListener('keydown',e=>{if(e.key==='Enter')choose(p.ADM_CD);});group.append(node);});typeLabels.forEach((label,i)=>{const row=document.createElement('span'),swatch=document.createElement('i');swatch.style.background=typeColors[i];row.append(swatch,label+' · '+props.filter(p=>kind(p)===i).length+'개 동');$('type-legend').append(row);});$('type-thresholds').textContent=`서울 전체 고정 기준: 밀도 ≥ ${n(s.density_q75)}명/km² / 접근성 ≤ ${s.access_q25.toFixed(3)}. 자치구를 바꿔도 기준은 같습니다.`;}
drawJoint();
(()=>{const colors=['#fff3e6','#fdd0a2','#fd8d3c','#d94801','#7f2704'],cuts=[1,10,25,50],group=el('g',{'class':'maparea'});$('far-map').append(group);
features.forEach(f=>{const p=f.properties,v=p.beyond_1km_pct,bin=cuts.filter(c=>v>c).length,node=el('path',{d:path(f),fill:colors[bin],'fill-rule':'evenodd','data-id':p.ADM_CD,tabindex:0,'aria-label':p.name});node.append(el('title',{},`${p.name}: 1km 밖 ${v.toFixed(1)}% (${n(p.beyond_1km_pop)}명)`));node.addEventListener('click',()=>choose(p.ADM_CD));node.addEventListener('keydown',e=>{if(e.key==='Enter')choose(p.ADM_CD);});group.append(node);});
const legend=$('far-legend');legend.append('1km 밖 인구 비율 ');['1% 이하','1~10%','10~25%','25~50%','50% 초과'].forEach((t,i)=>{const sw=document.createElement('i');sw.style.background=colors[i];legend.append(sw,t+' ');});
s.top_beyond.forEach(r=>{const tr=document.createElement('tr');[r.name,n(r.beyond_1km_pop)+'명',r.beyond_1km_pct.toFixed(1)+'%',r.access.toFixed(3)].forEach(t=>{const td=document.createElement('td');td.textContent=t;tr.append(td);});const id=props.find(p=>p.name===r.name).ADM_CD;tr.tabIndex=0;tr.addEventListener('click',()=>choose(id));tr.addEventListener('keydown',e=>{if(e.key==='Enter')choose(id);});$('far-table').append(tr);});})();
['density-map','access-map','joint-map'].forEach(id=>{const group=el('g',{'class':'overview-stations',display:'none'});window.CASE_STUDY.stations.forEach(st=>{const point=el('circle',{cx:ox+(st.x-minX)*scale,cy:oy+(maxY-st.y)*scale,r:2});point.append(el('title',{},`${st.name} / ${st.line}`));group.append(point);});$(id).append(group);});
$('show-stations').addEventListener('change',e=>document.querySelectorAll('.overview-stations').forEach(g=>g.setAttribute('display',e.target.checked?'inline':'none')));
typeSelect.addEventListener('change',()=>{highlight();$('detail').textContent=typeSelect.value===''?'전체 유형을 표시합니다.':typeLabels[Number(typeSelect.value)]+' 유형을 강조합니다. 다른 동은 흐리게 표시합니다.';});

// 사례 지도는 EPSG:5179 좌표와 사례 세 곳의 원본 격자만 사용한다. 외부 지도 서비스는 호출하지 않는다.
const caseStudy=window.CASE_STUDY,caseList=caseStudy.cases,caseSvg=$('case-map');
const caseTitles=['1km 밖 인구가 가장 적은 후보','1km 밖 인구 비율이 가장 높은 후보','밀도의 분모가 바꾸는 후보'];
let activeCase=0,selectedStation=caseList[0].default_station,caseView=[],caseInitial=[],dragState=null,dragMoved=false;
function polygonPath(geometry){if(!geometry)return '';const polys=geometry.type==='Polygon'?[geometry.coordinates]:geometry.type==='MultiPolygon'?geometry.coordinates:[];return polys.map(poly=>poly.map(ring=>ring.map((p,i)=>(i?'L':'M')+p[0].toFixed(2)+','+(-p[1]).toFixed(2)).join('')+'Z').join('')).join('');}
function viewCase(){caseSvg.setAttribute('viewBox',caseView.join(' '));}
function resetCaseView(){caseView=[...caseInitial];viewCase();}
function zoomCase(factor){const [x,y,w,h]=caseView,newW=Math.max(caseInitial[2]/8,Math.min(caseInitial[2],w*factor)),newH=newW*caseInitial[3]/caseInitial[2];caseView=[x+(w-newW)/2,y+(h-newH)/2,newW,newH];viewCase();}
function metric(label,value){const box=document.createElement('div'),title=document.createElement('span'),v=document.createElement('b');title.textContent=label;v.textContent=value;box.append(title,v);return box;}
function renderCase(index,keepView=false){activeCase=index;const c=caseList[index],m=c.metrics,defaultStation=c.stations.find(st=>st.key===selectedStation)||c.stations.find(st=>st.key===c.default_station);selectedStation=defaultStation.key;
if(!keepView){const x0=Math.min(c.bounds[0],defaultStation.x-1050),y0=Math.min(c.bounds[1],defaultStation.y-1050),x1=Math.max(c.bounds[2],defaultStation.x+1050),y1=Math.max(c.bounds[3],defaultStation.y+1050);caseInitial=[x0,-y1,x1-x0,y1-y0];caseView=[...caseInitial];}
caseSvg.replaceChildren();viewCase();caseSvg.setAttribute('aria-label',c.name+' 인구 격자와 역 위치 확대 지도');
const defs=el('defs'),pattern=el('pattern',{id:'missing-grid',width:55,height:55,patternUnits:'userSpaceOnUse'});pattern.append(el('rect',{width:55,height:55,fill:'#f4f4f0'}),el('path',{d:'M0 55L55 0',stroke:'#d2d5d0','stroke-width':9}));defs.append(pattern);caseSvg.append(defs);
caseSvg.append(el('path',{d:polygonPath(c.built),fill:'#dcebe0','fill-rule':'evenodd'}));
c.context.features.forEach(f=>{caseSvg.append(el('path',{d:polygonPath(f.geometry),fill:'none',stroke:'#c5cdc5','stroke-width':6}));});
c.grids.forEach(g=>{const p=g.properties,pop=p.population,color=!p.known?'url(#missing-grid)':pop<=0?'transparent':pop<100?'#f6dfa0':pop<300?'#dbad55':pop<600?'#b87932':'#784b24',node=el('path',{d:polygonPath(g.geometry),fill:color,stroke:pop>0?'#fff':'none','stroke-width':1,'class':'case-grid'});node.append(el('title',{},`${p.code} / ${p.known?n(pop)+'명':'인구 미수록'} / 보행망 ${n(p.distance)}m · 직선 ${n(p.line)}m`));node.addEventListener('click',()=>{if(dragMoved)return;selectedStation=p.station_key;renderCase(activeCase,true);const station=c.stations.find(st=>st.key===p.station_key);$('case-grid-detail').textContent=`선택 격자 ${p.code}: ${p.known?n(pop)+'명':'인구 미수록'} / 직선 최근접역 ${station.name}(${station.line})까지 직선 ${n(p.line)}m, 보행망 최근접역까지 ${n(p.distance)}m.`;});caseSvg.append(node);});
caseSvg.append(el('path',{d:polygonPath(c.boundary),fill:'none',stroke:'#18342e','stroke-width':13}));
{const rc=c.route.coords,end=rc[rc.length-1];caseSvg.append(el('path',{d:'M'+rc.map(p=>p[0].toFixed(1)+','+(-p[1]).toFixed(1)).join('L'),fill:'none',stroke:'#d94801','stroke-width':11,'stroke-linejoin':'round','pointer-events':'none'}));caseSvg.append(el('circle',{cx:end[0],cy:-end[1],r:22,fill:'#d94801',stroke:'#fff','stroke-width':5,'pointer-events':'none'}));}
[500,1000].forEach(radius=>{caseSvg.append(el('circle',{cx:defaultStation.x,cy:-defaultStation.y,r:radius,fill:'none',stroke:'#247b9f','stroke-width':7,'stroke-dasharray':radius===500?'30 20':'60 25','pointer-events':'none'}));caseSvg.append(el('text',{x:defaultStation.x+radius,y:-defaultStation.y-25,fill:'#247b9f','font-size':caseInitial[2]/65,'paint-order':'stroke',stroke:'#fff','stroke-width':8},'직선 '+radius+'m'));});
c.stations.forEach(st=>{const point=el('circle',{cx:st.x,cy:-st.y,r:st.key===selectedStation?24:14,fill:st.key===selectedStation?'#1c7772':'#18342e',stroke:'#fff','stroke-width':5,tabindex:0,'aria-label':st.name+' '+st.line});point.append(el('title',{},st.name+' / '+st.line));const select=()=>{selectedStation=st.key;renderCase(activeCase);$('case-grid-detail').textContent='거리 원 기준: '+st.name+' / '+st.line;};point.addEventListener('click',()=>{if(!dragMoved)select();});point.addEventListener('keydown',e=>{if(e.key==='Enter')select();});caseSvg.append(point);});
const fontSize=caseInitial[2]/58;
caseSvg.append(el('text',{x:defaultStation.x+35,y:-defaultStation.y-45,'font-size':fontSize,fill:'#18342e','paint-order':'stroke',stroke:'#fff','stroke-width':10,'class':'case-label'},defaultStation.name));
const bar=500,bx=caseInitial[0]+caseInitial[2]*.05,by=caseInitial[1]+caseInitial[3]*.92;caseSvg.append(el('path',{d:`M${bx} ${by}h${bar}`,stroke:'#18342e','stroke-width':12}));caseSvg.append(el('text',{x:bx,y:by-fontSize*.5,'font-size':fontSize,fill:'#18342e'},'500m'));
$('case-station').replaceChildren();c.stations.forEach(st=>{const option=new Option(st.name+' · '+st.line,st.key);option.selected=st.key===selectedStation;$('case-station').add(option);});
$('case-kicker').textContent='사례 '+(index+1)+' / '+caseTitles[index];$('case-title').textContent=c.name;
$('case-story').textContent=index===0?`고밀도·저접근성 후보이지만 보행망 기준 인구의 ${m.beyond_1km_pct.toFixed(1)}%만 최근접역에서 1km 밖에 있습니다. ‘하위 25%’라는 상대 순위가 ‘대부분의 주민이 역에서 1km 이상 멀다’는 뜻은 아닙니다.`:index===1?`보행망 기준 1km 밖에 사는 공개 인구가 ${m.beyond_1km_pct.toFixed(1)}%입니다. 후보 가운데 이 비율이 가장 높은 동으로, 첫 사례와 역까지의 절대 거리와 주민 분포가 다릅니다.`:`전체 면적 기준에서는 고밀도 후보가 아닙니다. 주거·상업·공업지역 면적만 분모로 쓰면 고밀도·저접근성 후보에 포함됩니다. 어떤 면적을 분모로 쓰는지가 결과에 영향을 줍니다.`;
$('case-metrics').replaceChildren(metric('인구가중 보행망 거리',n(m.mean_distance_m)+'m (직선 '+n(m.mean_distance_line_m)+'m)'),metric('1km 밖 공개 인구(보행망)',m.beyond_1km_pct.toFixed(1)+'%'),metric('전체 면적 기준 밀도',n(m.density)+'명/km²'),metric('개발지역 면적 기준 밀도',n(m.density_built)+'명/km²'));
$('case-density-note').textContent=`용도지역상 주거·상업·공업지역 비율 ${(m.built_share*100).toFixed(1)}%. 전체 면적 밀도 상위 25% 기준 ${n(s.density_q75)}명/km², 개발지역 면적 밀도 상위 25% 기준 ${n(caseStudy.density_built_q75)}명/km²입니다. 개발지역은 실제 건물이 있는 면적과 다릅니다.`;
$('case-ring-note').textContent=`기본 기준 역은 직선 최근접역으로 이 동의 공개 인구가 가장 많이 배정된 역·호선 좌표이며 보행망 최근접역과 다를 수 있습니다. 현재 선택: ${defaultStation.name} / ${defaultStation.line}. 원은 직선 반경입니다.`;
const rt=c.route,pt=rt.parts;$('case-route').textContent=`예시 격자 ${rt.grid_code}(공개 인구 ${n(rt.population)}명)의 보행 경로: 격자→노드 ${n(pt.grid_snap_m)}m + 보행망 ${n(pt.network_m)}m + 노드→역 ${n(pt.station_snap_m)}m = ${n(pt.walk_m)}m. 직선거리는 ${n(pt.line_m)}m, 도착 역은 ${rt.station_name}(${rt.station_line})입니다. 경로는 보행망 노드를 직선으로 이은 근사입니다.`;
document.querySelectorAll('.case-button').forEach((button,i)=>button.setAttribute('aria-pressed',i===index?'true':'false'));
}
caseList.forEach((c,index)=>{const button=document.createElement('button'),name=document.createElement('strong'),description=document.createElement('span');button.className='case-button';button.setAttribute('aria-pressed',index===0?'true':'false');name.textContent=c.name;description.textContent=caseTitles[index];button.append(name,description);button.addEventListener('click',()=>{selectedStation=c.default_station;renderCase(index);choose(c.id);$('case-grid-detail').textContent='격자를 누르면 공개 인구와 최근접역까지의 보행망·직선 거리를 확인할 수 있습니다.';});$('case-buttons').append(button);});
$('case-station').addEventListener('change',e=>{selectedStation=Number(e.target.value);renderCase(activeCase);$('case-grid-detail').textContent='선택한 역 기준 거리 원을 표시합니다.';});
$('case-zoom-in').addEventListener('click',()=>zoomCase(.65));$('case-zoom-out').addEventListener('click',()=>zoomCase(1/.65));$('case-reset').addEventListener('click',resetCaseView);
caseSvg.addEventListener('pointerdown',e=>{dragMoved=false;dragState={x:e.clientX,y:e.clientY,view:[...caseView]};});
caseSvg.addEventListener('pointermove',e=>{if(!dragState)return;const dx=e.clientX-dragState.x,dy=e.clientY-dragState.y;if(Math.hypot(dx,dy)<5&&!dragMoved)return;dragMoved=true;caseSvg.setPointerCapture(e.pointerId);const rect=caseSvg.getBoundingClientRect(),ratio=Math.max(dragState.view[2]/rect.width,dragState.view[3]/rect.height);caseView=[dragState.view[0]-dx*ratio,dragState.view[1]-dy*ratio,dragState.view[2],dragState.view[3]];viewCase();});
caseSvg.addEventListener('pointerup',()=>{dragState=null;});caseSvg.addEventListener('pointercancel',()=>{dragState=null;});
renderCase(0);

</script></body></html>"""
(DIST / "index.html").write_text(HTML)
print("로컬 앱:", (DIST / "index.html").relative_to(ROOT))